# UniGDNet-R — Stage 3 of 3
## Corrected resumable evaluation, statistics, figures, and reporting

Run this notebook only after the corrected Stage 2 training completes. It strictly rejects old-pipeline checkpoints, verifies the entire paper matrix, and caches every calibration/evaluation job atomically so an interrupted evaluation resumes without repeating finished work.

This revision uses a schema-aware five-class BraTS head (BG, NETC, SNFH, ET, RC), reports the official WT/TC/NETC/SNFH/ET/RC regions, uses deterministic calibration, takes HD95 spacing from the reference image grid, groups longitudinal scans by patient before splitting, and averages sessions and repeated seeds at patient level before statistical inference.


> **BraTS schema repair — 25 October 2026.** Modern BraTS labels are preserved as
> BG=0, NETC=1, SNFH=2, ET=3 and RC=4. Historical BraTS ET=4 is remapped to 3 only
> when the legacy modality filename family is detected. This is a scientifically breaking
> pipeline revision, so earlier preparation artefacts, checkpoints and evaluation caches are
> rejected or quarantined rather than mixed with corrected results. Longitudinal modern BraTS IDs are grouped by the patient prefix before splitting, while
> each timepoint remains a separate scan record and is averaged within patient for inference.


## 1. Environment setup

Use a clean Python 3.11 or 3.12 environment. Install an NVIDIA-compatible PyTorch build separately
from the official PyTorch selector if `torch.cuda.is_available()` is false. The notebook deliberately
does not replace an existing PyTorch installation.


In [ ]:
import os
import sys
import json
import math
import time
import random
import re
import hashlib
import platform
import subprocess
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import defaultdict, OrderedDict
from itertools import zip_longest
from typing import Dict, List, Tuple, Optional, Any

REQUIRED = [
    "numpy", "pandas", "nibabel", "scipy", "sklearn", "matplotlib",
    "seaborn", "tqdm", "psutil", "openpyxl"
]

def install_missing_packages():
    import importlib.util
    pip_names = {"sklearn": "scikit-learn"}
    missing = [pip_names.get(x, x) for x in REQUIRED if importlib.util.find_spec(x) is None]
    if missing:
        print("Installing:", missing)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

install_missing_packages()

import numpy as np
import pandas as pd
import nibabel as nib
from nibabel.processing import resample_from_to
import scipy.ndimage as ndi
from scipy.stats import wilcoxon
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import psutil

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

warnings.filterwarnings("ignore", category=UserWarning)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA runtime:", torch.version.cuda)
else:
    print("Install a CUDA-enabled PyTorch build from https://pytorch.org/get-started/locally/")

## 2. Configuration

`RUN_FULL_PAPER=False` performs a short pipeline check. Set it to `True` only after the dataset audit
and synthetic smoke test pass. The full protocol uses three random seeds and is intentionally long.


In [ ]:
@dataclass
class Config:
    data_root: str = r"D:\Datasets"
    output_root: str = r"D:\Papers\UniGDNetR"
    image_size: int = 256
    context_slices: int = 3
    base_channels: int = 24
    task_embedding_dim: int = 48
    dropout: float = 0.10
    batch_size: int = 2
    accumulation_steps: int = 4
    num_workers: int = 0  # safest on Windows with NIfTI caching
    positive_sampling_probability: float = 0.50
    modality_dropout_probability: float = 0.35
    neighbour_dropout_probability: float = 0.20
    learning_rate: float = 3e-4
    weight_decay: float = 1e-4
    warmup_epochs: int = 3
    full_epochs: int = 40
    smoke_epochs: int = 1
    patience: int = 10
    ema_decay: float = 0.999
    ce_weight: float = 0.45
    dice_weight: float = 0.45
    boundary_weight: float = 0.10
    gradient_clip: float = 5.0
    run_full_paper: bool = True
    full_seeds: Tuple[int, ...] = (13, 37, 71)
    smoke_seeds: Tuple[int, ...] = (13,)
    max_smoke_train_batches: int = 30
    max_smoke_val_batches: int = 20
    full_train_batches_per_epoch: int = 2500  # deterministic balanced compute budget
    full_val_batches_per_task: int = 200      # equal validation budget per task
    mc_dropout_samples: int = 8
    cache_cases: int = 6
    case_locality_block_size: int = 32  # same weighted draws, grouped to reduce NIfTI cache thrashing
    progress_every_batches: int = 25   # explicit heartbeat in addition to tqdm
    resume_every_epochs: int = 1       # atomic recovery checkpoint cadence
    checkpoint_every_batches: int = 2500  # periodic mid-epoch save (~10–25 minutes here)
    checkpoint_every_minutes: float = 20.0  # time fallback if data loading slows down

CFG = Config()
DATA_ROOT = Path(CFG.data_root)
OUTPUT_ROOT = Path(CFG.output_root)
for sub in ["manifests", "slice_indices", "checkpoints", "results", "figures", "tables", "logs"]:
    (OUTPUT_ROOT / sub).mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEEDS = CFG.full_seeds if CFG.run_full_paper else CFG.smoke_seeds
EPOCHS = CFG.full_epochs if CFG.run_full_paper else CFG.smoke_epochs
print(asdict(CFG))
print("Device:", DEVICE)
print("Mode:", "FULL PAPER" if CFG.run_full_paper else "SMOKE TEST")

In [ ]:
# Pipeline stage safety controls — do not remove.
PIPELINE_STAGE = "evaluate"
READ_ONLY_PREPARED_ARTIFACTS = True
REQUIRE_PREPARED_INDICES = True

print("Pipeline stage:", PIPELINE_STAGE)
print("Prepared artefacts are read-only:", READ_ONLY_PREPARED_ARTIFACTS)

In [ ]:
def seed_everything(seed: int = 13):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything(SEEDS[0])


## 3. Audit `D:\Datasets`

This cell records the actual local inventory in the results folder. It does not modify any dataset.


In [ ]:
def human_bytes(n: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    value = float(n)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f"{value:.2f} {unit}"
        value /= 1024

def audit_dataset_root(root: Path, max_depth: int = 2) -> pd.DataFrame:
    if not root.exists():
        raise FileNotFoundError(
            f"Dataset root not found: {root}. Confirm that the D: drive is mounted and update CFG.data_root."
        )
    rows = []
    for first in sorted([p for p in root.iterdir() if p.is_dir()]):
        files = [p for p in first.rglob("*") if p.is_file()]
        suffix_counts = defaultdict(int)
        total_bytes = 0
        for p in files:
            name = p.name.lower()
            suffix = ".nii.gz" if name.endswith(".nii.gz") else p.suffix.lower()
            suffix_counts[suffix or "[no suffix]"] += 1
            try:
                total_bytes += p.stat().st_size
            except OSError:
                pass
        rows.append({
            "folder": first.name,
            "path": str(first),
            "files": len(files),
            "size": human_bytes(total_bytes),
            "top_extensions": dict(sorted(suffix_counts.items(), key=lambda x: -x[1])[:8]),
        })
    return pd.DataFrame(rows)

inventory_df = audit_dataset_root(DATA_ROOT)
display(inventory_df)
inventory_path = OUTPUT_ROOT / "manifests" / "dataset_inventory.json"
if not READ_ONLY_PREPARED_ARTIFACTS:
    inventory_df.to_json(inventory_path, orient="records", indent=2)


## 4. Automatic discovery of BraTS, LiTS and ACDC-like NIfTI cases

The discovery logic supports common current and legacy naming conventions. Review the printed sample
records before training. If a local dataset has a custom layout, edit only its discovery function.


In [ ]:
PIPELINE_REVISION = "2026-10-25-brats-schema-rc-longitudinal-v7"
PREPARED_FORMAT_VERSION = 4

def is_nifti(p: Path) -> bool:
    n = p.name.lower()
    return n.endswith(".nii") or n.endswith(".nii.gz")

def nii_stem(p: Path) -> str:
    n = p.name
    return n[:-7] if n.lower().endswith(".nii.gz") else (n[:-4] if n.lower().endswith(".nii") else p.stem)

def safe_case_id(text: str) -> str:
    clean = "".join(ch if ch.isalnum() else "_" for ch in text).strip("_")
    digest = hashlib.sha1(text.encode("utf-8")).hexdigest()[:8]
    return f"{clean[:55]}_{digest}"

def find_named_root(parent: Path, keywords: Tuple[str, ...]) -> Optional[Path]:
    if not parent.exists():
        return None
    dirs = [parent] + [p for p in parent.rglob("*") if p.is_dir()]
    candidates = []
    for p in dirs:
        low = str(p).lower()
        score = sum(k.lower() in low for k in keywords)
        if score:
            candidates.append((score, len(p.parts), p))
    return sorted(candidates, key=lambda x: (-x[0], x[1]))[0][2] if candidates else parent


def _matches_modality_suffix(path: Path, aliases: Tuple[str, ...]) -> bool:
    """Match a complete modality suffix; never let ``-t2`` match ``-t2f``."""
    stem = nii_stem(path).lower()
    return any(re.search(rf"(?:^|[-_]){re.escape(alias.lower())}$", stem) for alias in aliases)


def _matched_modality_alias(path: Path, aliases: Tuple[str, ...]) -> str:
    matches = [alias for alias in aliases if _matches_modality_suffix(path, (alias,))]
    if len(matches) != 1:
        raise ValueError(f"Cannot determine the exact BraTS modality alias for {path}: {matches}")
    return matches[0]


def _brats_subject_id(seg: Path) -> str:
    stem = nii_stem(seg)
    subject = re.sub(r"(?:[-_](?:seg|segmentation))$", "", stem, flags=re.IGNORECASE)
    return subject or seg.parent.name


def _brats_patient_id(subject_id: str, brats_label_schema: str) -> str:
    """Remove only the standardized BraTS timepoint suffix.

    Modern IDs end in a three-digit timepoint/treatment code (for example,
    ``-000`` before treatment and ``-100``, ``-101`` ... after treatment).
    Keeping the complete subject ID as ``case_id`` preserves each scan, while
    grouping on the prefix keeps every timepoint from one patient in one split.
    Legacy IDs are left unchanged because their terminal digits identify the case.
    """
    if brats_label_schema == "modern_et3_rc4":
        match = re.fullmatch(
            r"(BraTS-[A-Za-z0-9]+-\d{5})-\d{3}",
            subject_id,
            flags=re.IGNORECASE,
        )
        if match:
            return match.group(1)
    return subject_id


def discover_brats(root: Path) -> List[dict]:
    if not root.exists():
        return []

    segs = [
        path for path in root.rglob("*")
        if path.is_file()
        and is_nifti(path)
        and re.search(r"(?:^|[-_])(?:seg|segmentation)$", nii_stem(path).lower())
    ]
    aliases = {
        "t1n": ("t1n", "t1"),
        "t1c": ("t1c", "t1ce", "t1gd"),
        "t2w": ("t2w", "t2"),
        "t2f": ("t2f", "flair"),
    }

    candidates = defaultdict(list)
    discovery_errors = []
    for seg in sorted(segs, key=lambda path: str(path).lower()):
        siblings = [
            path for path in seg.parent.iterdir()
            if path.is_file() and is_nifti(path) and path != seg
        ]
        matches_by_modality = {
            modality: sorted(
                [path for path in siblings if _matches_modality_suffix(path, modality_aliases)],
                key=lambda path: str(path).lower(),
            )
            for modality, modality_aliases in aliases.items()
        }
        # Ignore unrelated segmentation files elsewhere under the broad Brain root.
        # Once any BraTS modality is present, however, an incomplete/ambiguous set is fatal.
        if not any(matches_by_modality.values()):
            continue

        selected = []
        for modality in aliases:
            matches = matches_by_modality[modality]
            if len(matches) != 1:
                discovery_errors.append({
                    "segmentation": str(seg),
                    "modality": modality,
                    "matches": [str(path) for path in matches],
                })
                selected = []
                break
            selected.append(matches[0])

        if not selected:
            continue
        normalised_paths = {str(path.resolve()).casefold() for path in selected}
        if len(normalised_paths) != 4:
            discovery_errors.append({
                "segmentation": str(seg),
                "modality": "all",
                "matches": [str(path) for path in selected],
                "reason": "the four BraTS modalities are not four distinct files",
            })
            continue

        alias_by_modality = {
            modality: _matched_modality_alias(path, aliases[modality])
            for modality, path in zip(aliases, selected)
        }
        modern_aliases = {"t1n": "t1n", "t1c": "t1c", "t2w": "t2w", "t2f": "t2f"}
        is_modern = alias_by_modality == modern_aliases
        is_legacy = (
            alias_by_modality["t1n"] == "t1"
            and alias_by_modality["t1c"] in {"t1ce", "t1gd"}
            and alias_by_modality["t2w"] == "t2"
            and alias_by_modality["t2f"] == "flair"
        )
        if is_modern:
            # BraTS 2023+ convention: ET=3; post-treatment GLI may also contain RC=4.
            brats_label_schema = "modern_et3_rc4"
        elif is_legacy:
            # Historical BraTS convention: ET=4 and label 3 is unused.
            brats_label_schema = "legacy_et4"
        else:
            discovery_errors.append({
                "segmentation": str(seg),
                "modality": "naming_schema",
                "matches": alias_by_modality,
                "reason": "mixed modern/legacy BraTS modality naming is label-semantic ambiguous",
            })
            continue

        subject_id = _brats_subject_id(seg)
        patient_id = _brats_patient_id(subject_id, brats_label_schema)
        candidates[subject_id.casefold()].append({
            "case_id": safe_case_id(subject_id),
            "task": "brats",
            "group_id": safe_case_id(patient_id),
            "patient_id": patient_id,
            "subject_id": subject_id,
            "images": [str(path) for path in selected],
            "mask": str(seg),
            "modalities": ["T1n", "T1c", "T2w", "T2-FLAIR"],
            "brats_label_schema": brats_label_schema,
        })

    if discovery_errors:
        preview = discovery_errors[:20]
        raise RuntimeError(
            "BraTS discovery found missing, ambiguous, or duplicated modality assignments. "
            "Training is blocked rather than silently substituting a channel.\n"
            + json.dumps(preview, indent=2)
        )

    cases = []
    duplicate_subjects = {}
    for subject_key, records in sorted(candidates.items()):
        records = sorted(records, key=lambda item: item["mask"].casefold())
        cases.append(records[0])
        if len(records) > 1:
            duplicate_subjects[subject_key] = [item["mask"] for item in records]

    if duplicate_subjects:
        print(
            f"BraTS duplicate source copies excluded deterministically: {len(duplicate_subjects)} subject(s)."
        )
        duplicate_report = OUTPUT_ROOT / "manifests" / "brats_duplicate_sources.json"
        if not READ_ONLY_PREPARED_ARTIFACTS:
            temporary = duplicate_report.with_suffix(duplicate_report.suffix + f".{os.getpid()}.tmp")
            temporary.write_text(json.dumps(duplicate_subjects, indent=2), encoding="utf-8")
            os.replace(temporary, duplicate_report)

    return cases

def extract_lits_case_number(path: Path, prefix: str) -> Optional[int]:
    """Return an exact LiTS ID; case 0 can never match case 10 or 20."""
    stem = nii_stem(path).lower().strip()
    match = re.fullmatch(rf"{re.escape(prefix.lower())}[-_\s]*(\d+)", stem)
    return int(match.group(1)) if match else None

def discover_lits(root: Path) -> List[dict]:
    if not root.exists():
        return []

    files = [p for p in root.rglob("*") if p.is_file() and is_nifti(p)]
    volumes = defaultdict(list)
    masks = defaultdict(list)

    for path in files:
        volume_id = extract_lits_case_number(path, "volume")
        mask_id = extract_lits_case_number(path, "segmentation")
        if volume_id is not None:
            volumes[volume_id].append(path)
        if mask_id is not None:
            masks[mask_id].append(path)

    duplicate_volumes = {k: v for k, v in volumes.items() if len(v) != 1}
    duplicate_masks = {k: v for k, v in masks.items() if len(v) != 1}
    if duplicate_volumes or duplicate_masks:
        details = {
            "duplicate_volumes": {k: [str(p) for p in v] for k, v in duplicate_volumes.items()},
            "duplicate_masks": {k: [str(p) for p in v] for k, v in duplicate_masks.items()},
        }
        raise RuntimeError("Ambiguous duplicate LiTS identifiers:\n" + json.dumps(details, indent=2))

    paired_ids = sorted(set(volumes) & set(masks))
    unlabelled_volume_ids = sorted(set(volumes) - set(masks))
    orphan_mask_ids = sorted(set(masks) - set(volumes))

    if unlabelled_volume_ids:
        print("LiTS volumes without masks excluded from supervised training:", unlabelled_volume_ids)
    if orphan_mask_ids:
        print("LiTS masks without corresponding volumes:", orphan_mask_ids)
    if not paired_ids:
        raise RuntimeError(
            f"No exact volume-N/segmentation-N LiTS pairs were found below {root}."
        )

    cases = []
    for case_number in paired_ids:
        image = volumes[case_number][0]
        mask = masks[case_number][0]
        canonical_id = f"lits_{case_number:04d}"
        cases.append({
            "case_id": canonical_id,
            "task": "lits",
            "images": [str(image)],
            "group_id": canonical_id,
            "mask": str(mask),
            "modalities": ["CT"],
            "source_case_number": case_number,
        })
    return cases

def discover_acdc(root: Path) -> List[dict]:
    cases = []
    if not root.exists():
        return cases
    gts = [p for p in root.rglob("*") if p.is_file() and is_nifti(p) and "_gt" in nii_stem(p).lower()]
    for gt in gts:
        image_name = gt.name.replace("_gt.nii.gz", ".nii.gz").replace("_gt.nii", ".nii")
        image = gt.with_name(image_name)
        if image.exists():
            patient_name = nii_stem(gt).lower().split("_frame")[0]
            cases.append({
                "case_id": safe_case_id(str(gt)), "task": "acdc", "images": [str(image)],
                "group_id": safe_case_id(str(gt.parent / patient_name)),
                "mask": str(gt), "modalities": ["cine-MRI"],
            })
    unique = {c["mask"]: c for c in cases}
    return list(unique.values())

brain_root = DATA_ROOT / "Brain"
liver_root = DATA_ROOT / "Liver"
cardio_root = DATA_ROOT / "Cardio"

all_records = {
    "brats": discover_brats(brain_root),
    "lits": discover_lits(liver_root),
    "acdc": discover_acdc(cardio_root),
}

for task, records in all_records.items():
    print(f"{task:>6}: {len(records)} discovered cases")
    if records:
        print(json.dumps(records[0], indent=2))

missing_tasks = [t for t, records in all_records.items() if not records]
if missing_tasks:
    print("\nWARNING: no automatically compatible cases found for:", missing_tasks)
    print("Run the audit above and adjust the corresponding discovery function before full training.")

## 4.1 Mandatory image–mask integrity audit

This audit reads only NIfTI headers, so it is fast and does not load full volumes into
RAM. Training is blocked if any modality and mask have different voxel-array shapes.
Affine differences are reported separately for investigation.


In [ ]:
def canonical_header(path: str):
    return nib.as_closest_canonical(nib.load(path))

audit_rows = []
seen_case_ids = set()
duplicate_case_ids = []

for task, task_records in all_records.items():
    for record in task_records:
        case_id = record["case_id"]
        if case_id in seen_case_ids:
            duplicate_case_ids.append(case_id)
        seen_case_ids.add(case_id)

        normalised_images = [str(Path(path).resolve()).casefold() for path in record["images"]]
        if len(normalised_images) != len(set(normalised_images)):
            raise RuntimeError(
                f"A case assigns one file to multiple modalities: {task}/{case_id}: {record['images']}"
            )

        mask_header = canonical_header(record["mask"])
        mask_shape = tuple(mask_header.shape[:3])

        for modality_index, image_path in enumerate(record["images"]):
            image_header = canonical_header(image_path)
            image_shape = tuple(image_header.shape[:3])
            shape_match = image_shape == mask_shape
            affine_match = np.allclose(
                image_header.affine,
                mask_header.affine,
                atol=1e-3,
                rtol=1e-4,
            )
            audit_rows.append({
                "task": task,
                "case_id": case_id,
                "modality_index": modality_index,
                "image": image_path,
                "mask": record["mask"],
                "image_shape": str(image_shape),
                "mask_shape": str(mask_shape),
                "shape_match": shape_match,
                "affine_match": affine_match,
            })

        if task == "lits":
            image_id = extract_lits_case_number(Path(record["images"][0]), "volume")
            mask_id = extract_lits_case_number(Path(record["mask"]), "segmentation")
            if image_id != mask_id:
                raise RuntimeError(
                    f"LiTS numerical ID mismatch for {case_id}: volume {image_id}, mask {mask_id}"
                )

integrity_df = pd.DataFrame(audit_rows)
integrity_audit_path = OUTPUT_ROOT / "manifests" / "dataset_pairing_integrity_audit.csv"
if not READ_ONLY_PREPARED_ARTIFACTS:
    integrity_df.to_csv(integrity_audit_path, index=False)

if duplicate_case_ids:
    raise RuntimeError(f"Duplicate case IDs detected: {sorted(set(duplicate_case_ids))}")

shape_failures = integrity_df.loc[~integrity_df["shape_match"]]
if len(shape_failures):
    display(shape_failures)
    raise RuntimeError(
        "Image–mask shape mismatches remain. Training is blocked; do not crop or resize volumes to hide this error."
    )

affine_warnings = integrity_df.loc[~integrity_df["affine_match"]]
if len(affine_warnings):
    print(
        f"NOTICE: {len(affine_warnings)} image/mask grid differences will be "
        "aligned explicitly (linear image interpolation; nearest-neighbour mask interpolation)."
    )
    display(affine_warnings[["task", "case_id", "image", "mask"]])

print("Dataset integrity audit passed.")
print("Validated cases:", sum(len(v) for v in all_records.values()))
print("Validated image–mask combinations:", len(integrity_df))
display(integrity_df.groupby("task")[["shape_match", "affine_match"]].agg(["count", "mean"]))

## 5. Task specifications and patient-level splits

BraTS preserves the earlier GDNet five-logit label convention: background plus raw labels 1–4.
LiTS uses background/liver/tumour. ACDC uses background/RV/myocardium/LV.


In [ ]:
TASK_SPECS = {
    "brats": {"modalities": 4, "classes": 5, "class_names": ["BG", "NETC", "SNFH", "ET", "RC"]},
    "lits": {"modalities": 1, "classes": 3, "class_names": ["BG", "Liver", "Tumour"]},
    "acdc": {"modalities": 1, "classes": 4, "class_names": ["BG", "RV", "Myocardium", "LV"]},
}
TASK_SPECS = {k: v for k, v in TASK_SPECS.items() if all_records.get(k)}
assert TASK_SPECS, "No compatible segmentation datasets were discovered. Review Section 4."

def split_records(records: List[dict], seed: int = 13) -> Dict[str, List[dict]]:
    groups = sorted({r.get("group_id", r["case_id"]) for r in records})
    if len(groups) < 10:
        raise ValueError(f"At least 10 independent patients are required; found {len(groups)}")
    train_groups, temp_groups = train_test_split(groups, test_size=0.30, random_state=seed, shuffle=True)
    val_groups, test_groups = train_test_split(temp_groups, test_size=0.50, random_state=seed, shuffle=True)
    group_sets = {"train": set(train_groups), "val": set(val_groups), "test": set(test_groups)}
    split = {
        name: [r for r in records if r.get("group_id", r["case_id"]) in selected]
        for name, selected in group_sets.items()
    }
    assert not (group_sets["train"] & group_sets["val"] or group_sets["train"] & group_sets["test"]
                or group_sets["val"] & group_sets["test"]), "Patient leakage detected"
    return {
        "train": split["train"], "val": split["val"], "test": split["test"],
    }

splits = {task: split_records(records, SEEDS[0]) for task, records in all_records.items() if task in TASK_SPECS}
for task, task_splits in splits.items():
    record_counts = {name: len(records) for name, records in task_splits.items()}
    patient_counts = {
        name: len({record.get("group_id", record["case_id"]) for record in records})
        for name, records in task_splits.items()
    }
    print(task, "scan records", record_counts, "independent patients", patient_counts)
    if not READ_ONLY_PREPARED_ARTIFACTS:
        split_path = OUTPUT_ROOT / "manifests" / f"{task}_splits_seed{SEEDS[0]}.json"
        split_tmp = split_path.with_suffix(split_path.suffix + ".tmp")
        split_tmp.write_text(json.dumps(task_splits, indent=2), encoding="utf-8")
        os.replace(split_tmp, split_path)

## 6. NIfTI loading, normalisation and cached 2.5D slice indices


In [ ]:
class LRUCaseCache:
    def __init__(self, max_items: int = 6):
        self.max_items = max_items
        self.data = OrderedDict()

    def get(self, key, loader):
        if key in self.data:
            value = self.data.pop(key)
            self.data[key] = value
            return value
        value = loader()
        self.data[key] = value
        while len(self.data) > self.max_items:
            self.data.popitem(last=False)
        return value

    def clear(self):
        self.data.clear()

CASE_CACHE = LRUCaseCache(CFG.cache_cases)
REBUILT_INDEX_PATHS = []

def load_nifti_array(path: str, dtype=np.float32) -> np.ndarray:
    img = nib.as_closest_canonical(nib.load(path))
    arr = np.asarray(img.dataobj, dtype=dtype)
    if arr.ndim == 4 and arr.shape[-1] == 1:
        arr = arr[..., 0]
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D NIfTI, got {arr.shape}: {path}")
    return arr

def robust_normalise(arr: np.ndarray, task: str) -> np.ndarray:
    x = arr.astype(np.float32, copy=False)
    if task == "lits":
        x = np.clip(x, -200.0, 250.0)
        return ((x + 200.0) / 450.0 * 2.0 - 1.0).astype(np.float32)
    mask = np.isfinite(x) & (x != 0)
    if mask.sum() < 32:
        return np.nan_to_num(x, copy=False)
    lo, hi = np.percentile(x[mask], [0.5, 99.5])
    x = np.clip(x, lo, hi)
    mu, sd = float(x[mask].mean()), float(x[mask].std() + 1e-6)
    x = (x - mu) / sd
    x[~np.isfinite(x)] = 0
    return np.clip(x, -5, 5).astype(np.float32)


def standardise_mask_labels(mask: np.ndarray, task: str, source: str = "",
                            brats_label_schema: Optional[str] = None) -> np.ndarray:
    """Return contiguous labels while preserving edition-specific BraTS semantics.

    Modern BraTS (2023+) uses 1=NETC, 2=SNFH, 3=ET and, for post-treatment
    GLI, 4=RC. Historical BraTS uses 1=NCR/NET, 2=ED, 4=ET with label 3
    unused; only that explicitly detected legacy schema maps 4 -> 3.
    """
    if not np.isfinite(mask).all():
        raise ValueError(f"Mask contains non-finite values: {source}")
    rounded = np.rint(mask).astype(np.int16, copy=False)
    if not np.allclose(mask, rounded, atol=1e-3, rtol=0):
        raise ValueError(f"Mask contains non-integer label values: {source}")
    labels = set(int(value) for value in np.unique(rounded))
    if task == "brats":
        if brats_label_schema == "modern_et3_rc4":
            if not labels.issubset({0, 1, 2, 3, 4}):
                raise ValueError(
                    f"Unexpected modern BraTS labels {sorted(labels)}: {source}"
                )
            return rounded
        if brats_label_schema == "legacy_et4":
            if not labels.issubset({0, 1, 2, 4}):
                raise ValueError(
                    f"Unexpected legacy BraTS labels {sorted(labels)}: {source}"
                )
            if 4 in labels:
                rounded = rounded.copy()
                rounded[rounded == 4] = 3
            return rounded
        raise ValueError(
            f"Missing or unknown BraTS label schema {brats_label_schema!r}: {source}"
        )

    expected = TASK_SPECS[task]["classes"]
    if rounded.min() < 0 or rounded.max() >= expected:
        raise ValueError(
            f"Unexpected labels {sorted(labels)} for task {task}; expected 0..{expected - 1}: {source}"
        )
    return rounded


def _same_grid(left, right) -> bool:
    return (
        tuple(left.shape[:3]) == tuple(right.shape[:3])
        and np.allclose(left.affine, right.affine, atol=1e-3, rtol=1e-4)
    )


def load_record_mask(record: dict) -> np.ndarray:
    reference = nib.as_closest_canonical(nib.load(record["images"][0]))
    mask_image = nib.as_closest_canonical(nib.load(record["mask"]))
    if not _same_grid(mask_image, reference):
        mask_image = resample_from_to(mask_image, reference, order=0)
    mask = np.asarray(mask_image.dataobj, dtype=np.float32)
    if mask.ndim == 4 and mask.shape[-1] == 1:
        mask = mask[..., 0]
    if mask.ndim != 3:
        raise ValueError(f"Expected a 3D mask, got {mask.shape}: {record['mask']}")
    return standardise_mask_labels(
        mask,
        record["task"],
        record["mask"],
        brats_label_schema=record.get("brats_label_schema"),
    )


def load_case(record: dict) -> Tuple[np.ndarray, np.ndarray]:
    key = (
        tuple(record["images"]), record["mask"], record["task"],
        record.get("brats_label_schema"), PIPELINE_REVISION,
    )

    def _loader():
        reference = nib.as_closest_canonical(nib.load(record["images"][0]))
        image_arrays = []
        for path in record["images"]:
            image = nib.as_closest_canonical(nib.load(path))
            if not _same_grid(image, reference):
                image = resample_from_to(image, reference, order=1)
            array = np.asarray(image.dataobj, dtype=np.float32)
            if array.ndim == 4 and array.shape[-1] == 1:
                array = array[..., 0]
            if array.ndim != 3:
                raise ValueError(f"Expected 3D NIfTI, got {array.shape}: {path}")
            image_arrays.append(robust_normalise(array, record["task"]))

        mask = load_record_mask(record)
        shape = image_arrays[0].shape
        if any(array.shape != shape for array in image_arrays) or mask.shape != shape:
            raise ValueError(
                "Shape mismatch after explicit canonical-grid resampling. "
                f"case={record['case_id']}; image_shapes={[x.shape for x in image_arrays]}; "
                f"mask_shape={mask.shape}"
            )
        return np.stack(image_arrays, axis=0), mask

    return CASE_CACHE.get(key, _loader)

def file_signature(path: str) -> dict:
    p = Path(path)
    stat = p.stat()
    return {
        "path": str(p.resolve()),
        "size": int(stat.st_size),
        "mtime_ns": int(stat.st_mtime_ns),
    }

def records_fingerprint(records: List[dict]) -> str:
    payload = []
    for record in sorted(records, key=lambda r: (r["task"], r["case_id"])):
        payload.append({
            "task": record["task"],
            "case_id": record["case_id"],
            "group_id": record.get("group_id", record["case_id"]),
            "patient_id": record.get("patient_id"),
            "subject_id": record.get("subject_id"),
            "brats_label_schema": record.get("brats_label_schema"),
            "images": [file_signature(path) for path in record["images"]],
            "mask": file_signature(record["mask"]),
        })
    serialised = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(serialised.encode("utf-8")).hexdigest()

TASK_DATASET_FINGERPRINTS = {
    task: records_fingerprint(records)
    for task, records in all_records.items()
    if task in TASK_SPECS
}

def fingerprint_for_tasks(tasks) -> str:
    selected = {
        task: TASK_DATASET_FINGERPRINTS[task]
        for task in sorted(tasks)
    }
    serialised = json.dumps(
        {"pipeline_revision": PIPELINE_REVISION, "tasks": selected},
        sort_keys=True,
        separators=(",", ":"),
    )
    return hashlib.sha256(serialised.encode("utf-8")).hexdigest()

DATASET_FINGERPRINT = fingerprint_for_tasks(sorted(TASK_SPECS))
fingerprint_manifest = {
    "pipeline_revision": PIPELINE_REVISION,
    "combined_fingerprint": DATASET_FINGERPRINT,
    "task_fingerprints": TASK_DATASET_FINGERPRINTS,
}
fingerprint_path = OUTPUT_ROOT / "manifests" / "dataset_fingerprints.json"
if not READ_ONLY_PREPARED_ARTIFACTS:
    fingerprint_tmp = fingerprint_path.with_suffix(fingerprint_path.suffix + ".tmp")
    fingerprint_tmp.write_text(json.dumps(fingerprint_manifest, indent=2), encoding="utf-8")
    os.replace(fingerprint_tmp, fingerprint_path)
print("Corrected dataset fingerprint:", DATASET_FINGERPRINT)

def build_slice_index(records: List[dict], split_name: str, task: str) -> pd.DataFrame:
    split_fingerprint = records_fingerprint(records)[:12]
    path = OUTPUT_ROOT / "slice_indices" / f"{task}_{split_name}_{split_fingerprint}.csv"
    expected_case_ids = {record["case_id"] for record in records}

    if path.exists():
        cached = pd.read_csv(path)
        required_columns = {"case_id", "group_id", "z", "positive", "labels"}
        cached_case_ids = set(cached["case_id"].astype(str).unique()) if "case_id" in cached else set()
        cache_is_complete = (
            required_columns.issubset(cached.columns)
            and cached_case_ids == expected_case_ids
            and len(cached) > 0
            and not cached.duplicated(["case_id", "z"]).any()
        )
        if cache_is_complete:
            print("Using fingerprint-matched slice index:", path.name)
            return cached
        print("Ignoring incompatible cached slice index:", path.name)

    if REQUIRE_PREPARED_INDICES:
        raise FileNotFoundError(
            f"Prepared slice index is missing or incompatible: {path}. "
            "Run 01_UniGDNetR_PREPARE.ipynb successfully before training or evaluation."
        )

    rows = []
    for record in tqdm(records, desc=f"Indexing {task}/{split_name}"):
        mask = load_record_mask(record)
        for z in range(mask.shape[2]):
            labels, counts = np.unique(mask[:, :, z], return_counts=True)
            rows.append({
                "case_id": record["case_id"],
                "group_id": record.get("group_id", record["case_id"]),
                "z": z,
                "positive": int(np.any(mask[:, :, z] > 0)),
                "labels": json.dumps({int(a): int(b) for a, b in zip(labels, counts)}),
            })
    df = pd.DataFrame(rows)
    temporary = path.with_suffix(path.suffix + f".{os.getpid()}.tmp")
    df.to_csv(temporary, index=False)
    os.replace(temporary, path)
    REBUILT_INDEX_PATHS.append(str(path))
    return df

In [ ]:
# Strict hand-off: evaluation never rebuilds data artefacts or mixes pipeline revisions.
prepared_marker_path = OUTPUT_ROOT / "manifests" / "unigdnetr_prepared.json"
if not prepared_marker_path.exists():
    raise FileNotFoundError(
        f"Preparation marker not found: {prepared_marker_path}. Run the corrected Stage 2 notebook first."
    )

prepared_payload = json.loads(prepared_marker_path.read_text(encoding="utf-8"))
preflight_problems = []
if int(prepared_payload.get("prepared_format_version", -1)) != PREPARED_FORMAT_VERSION:
    preflight_problems.append("prepared format version differs")
if prepared_payload.get("pipeline_revision") != PIPELINE_REVISION:
    preflight_problems.append("pipeline revision differs")
if prepared_payload.get("dataset_fingerprint") != DATASET_FINGERPRINT:
    preflight_problems.append("dataset fingerprint differs")
if sorted(prepared_payload.get("tasks", [])) != sorted(TASK_SPECS):
    preflight_problems.append("task set differs")
if int(prepared_payload.get("split_seed", -1)) != int(SEEDS[0]):
    preflight_problems.append("split seed differs")
missing_indices = [path for path in prepared_payload.get("index_files", []) if not Path(path).exists()]
if missing_indices:
    preflight_problems.append(f"{len(missing_indices)} prepared slice index file(s) are missing")

if preflight_problems:
    raise RuntimeError(
        "Stage 3 cannot mix old and corrected data artefacts: "
        + "; ".join(preflight_problems)
        + ". Run the corrected Stage 2 notebook first; it refreshes preparation automatically."
    )

print(
    "PREPARATION VERIFIED |", prepared_payload.get("prepared_utc"),
    "| tasks=", sorted(TASK_SPECS),
    "| indices=", len(prepared_payload.get("index_files", [])),
    flush=True,
)

In [ ]:
class VolumeSliceDataset(Dataset):
    def __init__(self, records: List[dict], index_df: pd.DataFrame, training: bool):
        self.records = {r["case_id"]: r for r in records}
        self.df = index_df.reset_index(drop=True)
        self.training = training
        self.k = CFG.context_slices
        assert self.k % 2 == 1

    def __len__(self):
        return len(self.df)

    def _extract(self, volume: np.ndarray, z: int) -> np.ndarray:
        radius = self.k // 2
        indices = [min(max(z + dz, 0), volume.shape[-1] - 1) for dz in range(-radius, radius + 1)]
        channels = [volume[m, :, :, zi] for m in range(volume.shape[0]) for zi in indices]
        return np.stack(channels, axis=0).astype(np.float32)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        record = self.records[row.case_id]
        volume, mask = load_case(record)
        x = torch.from_numpy(self._extract(volume, int(row.z)))
        y = torch.from_numpy(mask[:, :, int(row.z)].astype(np.int64))
        x = F.interpolate(x[None], size=(CFG.image_size, CFG.image_size), mode="bilinear", align_corners=False)[0]
        y = F.interpolate(y[None, None].float(), size=(CFG.image_size, CFG.image_size), mode="nearest")[0, 0].long()

        if self.training:
            if random.random() < 0.5:
                x, y = torch.flip(x, [-1]), torch.flip(y, [-1])
            if random.random() < 0.5:
                x, y = torch.flip(x, [-2]), torch.flip(y, [-2])
            rotations = random.randint(0, 3)
            if rotations:
                x, y = torch.rot90(x, rotations, [-2, -1]), torch.rot90(y, rotations, [-2, -1])
            if random.random() < 0.35:
                x = x * random.uniform(0.90, 1.10) + random.uniform(-0.08, 0.08)
            if random.random() < 0.25:
                x = x + torch.randn_like(x) * random.uniform(0.0, 0.04)

            modalities = TASK_SPECS[record["task"]]["modalities"]
            if modalities > 1 and random.random() < CFG.modality_dropout_probability:
                keep = torch.rand(modalities) > 0.35
                if not keep.any():
                    keep[random.randrange(modalities)] = True
                for m in range(modalities):
                    if not keep[m]:
                        x[m*self.k:(m+1)*self.k] = 0
            if random.random() < CFG.neighbour_dropout_probability:
                for m in range(modalities):
                    if random.random() < 0.5:
                        x[m*self.k] = 0
                    if random.random() < 0.5:
                        x[m*self.k + self.k - 1] = 0

        return {"image": x, "mask": y, "task": record["task"], "case_id": record["case_id"], "z": int(row.z)}


class CaseLocalWeightedSampler:
    """Deterministic weighted sampling, emitted in cache-friendly patient blocks.

    ``start_sample`` enables exact, fast mid-epoch restoration: the sampler rebuilds
    the same deterministic draw order and starts yielding at the saved position. No
    completed NIfTI samples are loaded merely to reach the restore point.
    """

    def __init__(self, index_df: pd.DataFrame, positive_probability: float,
                 block_size: int = 32, seed: int = 13):
        positive = index_df["positive"].to_numpy(dtype=bool)
        n_pos, n_neg = max(int(positive.sum()), 1), max(int((~positive).sum()), 1)
        self.weights = torch.as_tensor(
            np.where(positive, positive_probability / n_pos, (1 - positive_probability) / n_neg),
            dtype=torch.double,
        )
        self.case_ids = index_df["case_id"].astype(str).to_numpy()
        self.num_samples = len(index_df)
        self.block_size = max(int(block_size), 1)
        self.base_seed = int(seed)
        self.epoch = 0
        self.run_seed = int(seed)
        self.start_sample = 0

    def __len__(self):
        # Keep the full epoch length stable for plans/progress displays. The iterator
        # itself yields only the uncompleted suffix after a mid-epoch restore.
        return self.num_samples

    def set_epoch(self, epoch: int, run_seed: int, start_sample: int = 0):
        start_sample = int(start_sample)
        if not 0 <= start_sample <= self.num_samples:
            raise ValueError(
                f"Invalid sampler restore position {start_sample}; expected 0..{self.num_samples}."
            )
        self.epoch = int(epoch)
        self.run_seed = int(run_seed)
        self.start_sample = start_sample

    def __iter__(self):
        combined_seed = self.run_seed * 1_000_003 + self.epoch * 10_007 + self.base_seed
        generator = torch.Generator()
        generator.manual_seed(combined_seed)
        sampled = torch.multinomial(
            self.weights,
            self.num_samples,
            replacement=True,
            generator=generator,
        ).tolist()

        rng = random.Random(combined_seed)
        by_case = defaultdict(list)
        for index in sampled:
            by_case[self.case_ids[index]].append(index)

        blocks = []
        for indices in by_case.values():
            rng.shuffle(indices)
            blocks.extend(
                indices[start:start + self.block_size]
                for start in range(0, len(indices), self.block_size)
            )
        rng.shuffle(blocks)

        skipped = 0
        for block in blocks:
            block_end = skipped + len(block)
            if block_end <= self.start_sample:
                skipped = block_end
                continue
            first = max(self.start_sample - skipped, 0)
            yield from block[first:]
            skipped = block_end


def make_sampler(index_df: pd.DataFrame) -> CaseLocalWeightedSampler:
    return CaseLocalWeightedSampler(
        index_df,
        positive_probability=CFG.positive_sampling_probability,
        block_size=CFG.case_locality_block_size,
        seed=SEEDS[0],
    )


def _evenly_spaced_indices(indices: List[int], count: int) -> List[int]:
    """Select a deterministic spread across an ordered slice list."""
    indices = list(indices)
    count = min(max(int(count), 0), len(indices))
    if count == 0:
        return []
    if count == len(indices):
        return indices
    positions = np.linspace(0, len(indices) - 1, num=count)
    return [indices[int(round(position))] for position in positions]


def validation_subset_indices(index_df: pd.DataFrame, max_batches: int) -> List[int]:
    """Build a fixed patient-covered, foreground/background-balanced validation subset."""
    frame = index_df.reset_index(drop=True)
    max_samples = min(len(frame), max(int(max_batches), 1) * CFG.batch_size)
    if max_samples >= len(frame):
        return list(range(len(frame)))

    all_cases = sorted(frame["case_id"].astype(str).unique())
    if max_samples < len(all_cases):
        positions = np.linspace(0, len(all_cases) - 1, num=max_samples)
        case_names = [all_cases[int(round(position))] for position in positions]
    else:
        case_names = all_cases

    case_rows = {
        case_name: frame.index[frame["case_id"].astype(str) == case_name].tolist()
        for case_name in case_names
    }
    quotas = {case_name: 1 for case_name in case_names}
    remaining = max_samples - len(case_names)
    while remaining:
        progressed = False
        for case_name in case_names:
            if quotas[case_name] < len(case_rows[case_name]):
                quotas[case_name] += 1
                remaining -= 1
                progressed = True
                if remaining == 0:
                    break
        if not progressed:
            raise RuntimeError("Validation subset allocation exhausted all patient slices unexpectedly.")

    selected = []
    for case_name in case_names:
        quota = quotas[case_name]
        rows = case_rows[case_name]
        positive = [index for index in rows if bool(frame.at[index, "positive"])]
        negative = [index for index in rows if not bool(frame.at[index, "positive"])]

        positive_count = min(len(positive), (quota + 1) // 2)
        negative_count = min(len(negative), quota - positive_count)
        if positive_count + negative_count < quota:
            positive_count += min(len(positive) - positive_count, quota - positive_count - negative_count)
        if positive_count + negative_count < quota:
            negative_count += min(len(negative) - negative_count, quota - positive_count - negative_count)

        chosen = _evenly_spaced_indices(positive, positive_count)
        chosen += _evenly_spaced_indices(negative, negative_count)
        selected.extend(sorted(chosen, key=lambda index: int(frame.at[index, "z"])))

    if len(selected) != max_samples or len(selected) != len(set(selected)):
        raise RuntimeError(
            f"Validation subset construction failed: selected={len(selected)}, expected={max_samples}."
        )
    return selected


def build_loaders(task_splits: Dict[str, Dict[str, List[dict]]]) -> Dict[str, Dict[str, DataLoader]]:
    loaders = defaultdict(dict)
    for task, parts in task_splits.items():
        for split_name, records in parts.items():
            idx_df = build_slice_index(records, split_name, task)
            ds = VolumeSliceDataset(records, idx_df, training=(split_name == "train"))
            kwargs = dict(
                batch_size=CFG.batch_size,
                num_workers=CFG.num_workers,
                pin_memory=torch.cuda.is_available(),
                persistent_workers=CFG.num_workers > 0,
            )
            if split_name == "train":
                loaders[task][split_name] = DataLoader(ds, sampler=make_sampler(idx_df), **kwargs)
            elif split_name == "val":
                validation_cap = (
                    CFG.full_val_batches_per_task if CFG.run_full_paper
                    else CFG.max_smoke_val_batches
                )
                subset_indices = validation_subset_indices(idx_df, validation_cap)
                # Subset rows stay grouped by patient, preserving NIfTI cache locality.
                loaders[task][split_name] = DataLoader(
                    Subset(ds, subset_indices), shuffle=False, **kwargs
                )
                print(
                    f"Validation coverage {task}: {len(subset_indices):,}/{len(idx_df):,} slices "
                    f"across {idx_df.iloc[subset_indices]['case_id'].nunique()} patient(s).",
                    flush=True,
                )
            else:
                loaders[task][split_name] = DataLoader(ds, shuffle=False, **kwargs)
    return dict(loaders)


loaders = build_loaders(splits)
loader_batch_counts = {
    task: {part: len(loader) for part, loader in value.items()}
    for task, value in loaders.items()
}
print("Loader batches per epoch:", loader_batch_counts, flush=True)
print(
    "Training sampler: weighted draws in case-local blocks of",
    CFG.case_locality_block_size,
    "slices.",
    flush=True,
)


## 7. UniGDNet-R architecture

The reliability gate receives mean and standard-deviation descriptors from each neighbouring slice.
It learns soft weights over the three-slice context and returns the centre image plus a weighted
context residual. Task-specific stems reconcile different modality counts. All subsequent encoder and
decoder blocks are shared and conditioned by a learned task embedding through FiLM.


In [ ]:
class ReliabilityGatedSliceFusion(nn.Module):
    def __init__(self, modalities: int, k: int = 3, adaptive: bool = True):
        super().__init__()
        self.modalities, self.k, self.adaptive = modalities, k, adaptive
        self.gate = nn.Sequential(nn.Linear(2, 16), nn.SiLU(), nn.Linear(16, 1))

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        b, c, h, w = x.shape
        if c != self.modalities * self.k:
            raise ValueError(f"Expected {self.modalities*self.k} channels, received {c}")
        xv = x.view(b, self.modalities, self.k, h, w)
        stats = torch.stack([xv.mean((-1, -2)), xv.std((-1, -2), unbiased=False)], dim=-1)
        if self.adaptive:
            scores = self.gate(stats).squeeze(-1)
            missing = xv.abs().sum((-1, -2)) < 1e-7
            scores = scores.masked_fill(missing, -20.0)
            weights = torch.softmax(scores, dim=-1)
        else:
            weights = torch.full((b, self.modalities, self.k), 1.0 / self.k, device=x.device, dtype=x.dtype)
        context = (xv * weights[..., None, None]).sum(dim=2)
        centre = xv[:, :, self.k // 2]
        fused = torch.cat([centre, context - centre], dim=1)
        return fused, weights

def group_count(channels: int) -> int:
    for g in (8, 4, 2, 1):
        if channels % g == 0:
            return g
    return 1

class FiLMResBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, emb_dim: int, dropout: float, use_film: bool = True):
        super().__init__()
        self.use_film = use_film
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(group_count(out_ch), out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(group_count(out_ch), out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()
        self.film = nn.Linear(emb_dim, 2 * out_ch) if use_film else None
        self.dropout = nn.Dropout2d(dropout)

    def forward(self, x: torch.Tensor, emb: torch.Tensor) -> torch.Tensor:
        h = self.gn1(self.conv1(x))
        if self.use_film:
            gamma, beta = self.film(emb).chunk(2, dim=1)
            h = h * (1 + gamma[:, :, None, None]) + beta[:, :, None, None]
        h = F.silu(h)
        h = self.dropout(h)
        h = self.gn2(self.conv2(h))
        return F.silu(h + self.skip(x))

class UniGDNetR(nn.Module):
    def __init__(self, task_specs: dict, base: int = 24, emb_dim: int = 48,
                 dropout: float = 0.10, adaptive_slice: bool = True, use_film: bool = True):
        super().__init__()
        self.task_specs = task_specs
        self.task_names = sorted(task_specs)
        self.task_to_idx = {t: i for i, t in enumerate(self.task_names)}
        self.task_embedding = nn.Embedding(len(self.task_names), emb_dim)
        self.fusions = nn.ModuleDict({
            t: ReliabilityGatedSliceFusion(s["modalities"], CFG.context_slices, adaptive_slice)
            for t, s in task_specs.items()
        })
        self.stems = nn.ModuleDict({
            t: nn.Sequential(
                nn.Conv2d(2*s["modalities"], base, 3, padding=1, bias=False),
                nn.GroupNorm(group_count(base), base), nn.SiLU()
            ) for t, s in task_specs.items()
        })
        self.e1 = FiLMResBlock(base, base, emb_dim, dropout, use_film)
        self.d1 = nn.Conv2d(base, base*2, 3, stride=2, padding=1, bias=False)
        self.e2 = FiLMResBlock(base*2, base*2, emb_dim, dropout, use_film)
        self.d2 = nn.Conv2d(base*2, base*4, 3, stride=2, padding=1, bias=False)
        self.e3 = FiLMResBlock(base*4, base*4, emb_dim, dropout, use_film)
        self.d3 = nn.Conv2d(base*4, base*6, 3, stride=2, padding=1, bias=False)
        self.bottleneck = FiLMResBlock(base*6, base*6, emb_dim, dropout, use_film)
        self.u3 = nn.ConvTranspose2d(base*6, base*4, 2, stride=2)
        self.dec3 = FiLMResBlock(base*8, base*4, emb_dim, dropout, use_film)
        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2)
        self.dec2 = FiLMResBlock(base*4, base*2, emb_dim, dropout, use_film)
        self.u1 = nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.dec1 = FiLMResBlock(base*2, base, emb_dim, dropout, use_film)
        self.heads = nn.ModuleDict({t: nn.Conv2d(base, s["classes"], 1) for t, s in task_specs.items()})

    def forward(self, x: torch.Tensor, task: str, return_aux: bool = False):
        if task not in self.task_specs:
            raise KeyError(task)
        task_idx = torch.full((x.shape[0],), self.task_to_idx[task], device=x.device, dtype=torch.long)
        emb = self.task_embedding(task_idx)
        fused, weights = self.fusions[task](x)
        x0 = self.stems[task](fused)
        e1 = self.e1(x0, emb)
        e2 = self.e2(self.d1(e1), emb)
        e3 = self.e3(self.d2(e2), emb)
        b = self.bottleneck(self.d3(e3), emb)
        x = self.dec3(torch.cat([self.u3(b), e3], dim=1), emb)
        x = self.dec2(torch.cat([self.u2(x), e2], dim=1), emb)
        x = self.dec1(torch.cat([self.u1(x), e1], dim=1), emb)
        logits = self.heads[task](x)
        return (logits, {"slice_weights": weights, "bottleneck": b}) if return_aux else logits

def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Architecture definitions loaded; models will be created only when needed.")


## 8. Losses and slice-level metrics


In [ ]:
def soft_dice_loss(logits: torch.Tensor, target: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    n_classes = logits.shape[1]
    probs = torch.softmax(logits, dim=1)
    onehot = F.one_hot(target.clamp(0, n_classes-1), n_classes).permute(0, 3, 1, 2).float()
    probs, onehot = probs[:, 1:], onehot[:, 1:]
    dims = (0, 2, 3)
    intersection = (probs * onehot).sum(dims)
    denom = probs.sum(dims) + onehot.sum(dims)
    present = onehot.sum(dims) > 0
    dice = (2 * intersection + eps) / (denom + eps)
    return 1 - (dice[present].mean() if present.any() else dice.mean())

def sobel_edges(x: torch.Tensor) -> torch.Tensor:
    c = x.shape[1]
    kx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], device=x.device, dtype=x.dtype).view(1,1,3,3)
    ky = kx.transpose(-1, -2)
    kx, ky = kx.repeat(c,1,1,1), ky.repeat(c,1,1,1)
    gx = F.conv2d(x, kx, padding=1, groups=c)
    gy = F.conv2d(x, ky, padding=1, groups=c)
    return torch.sqrt(gx.square() + gy.square() + 1e-6)

def boundary_consistency_loss(logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    n_classes = logits.shape[1]
    probs = torch.softmax(logits, dim=1)[:, 1:]
    onehot = F.one_hot(target.clamp(0, n_classes-1), n_classes).permute(0,3,1,2).float()[:, 1:]
    return F.l1_loss(sobel_edges(probs), sobel_edges(onehot))

class CompoundSegmentationLoss(nn.Module):
    def __init__(self, use_boundary: bool = True):
        super().__init__()
        self.use_boundary = use_boundary

    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target)
        dice = soft_dice_loss(logits, target)
        boundary = boundary_consistency_loss(logits, target) if self.use_boundary else logits.new_tensor(0.0)
        total = CFG.ce_weight*ce + CFG.dice_weight*dice + (CFG.boundary_weight*boundary if self.use_boundary else 0)
        return total, {"ce": float(ce.detach()), "dice_loss": float(dice.detach()), "boundary": float(boundary.detach())}

@torch.no_grad()
def mean_foreground_dice(logits: torch.Tensor, target: torch.Tensor) -> float:
    pred = logits.argmax(1)
    vals = []
    for c in range(1, logits.shape[1]):
        p, y = pred == c, target == c
        # Penalise false-positive classes on negative slices; skip only a true-empty pair.
        if y.any() or p.any():
            vals.append(float((2*(p & y).sum() + 1e-6) / (p.sum() + y.sum() + 1e-6)))
    return float(np.mean(vals)) if vals else float("nan")


## 10. Training utilities: EMA, warm-up + cosine, AMP and early stopping


In [ ]:
RESUME_ENGINE_REVISION = "2026-10-25-mid-epoch-resume-stale-lock-v5"


TRAINING_CONFIG_KEYS = [
    "image_size", "context_slices", "base_channels", "task_embedding_dim", "dropout",
    "batch_size", "accumulation_steps", "positive_sampling_probability",
    "modality_dropout_probability", "neighbour_dropout_probability", "learning_rate",
    "weight_decay", "warmup_epochs", "full_epochs", "smoke_epochs", "patience",
    "ema_decay", "ce_weight", "dice_weight", "boundary_weight", "gradient_clip",
    "run_full_paper", "num_workers", "case_locality_block_size",
    "full_train_batches_per_epoch", "full_val_batches_per_task",
]


class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1-self.decay)
            else:
                self.shadow[k].copy_(v)

    def copy_to(self, model: nn.Module):
        model.load_state_dict(self.shadow, strict=True)


def torch_load_compat(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def make_scheduler(optimizer, total_epochs: int):
    def factor(epoch):
        if epoch < CFG.warmup_epochs:
            return max((epoch + 1) / max(CFG.warmup_epochs, 1), 0.05)
        progress = (epoch - CFG.warmup_epochs) / max(total_epochs - CFG.warmup_epochs, 1)
        return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, factor)


def autocast_context():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda")


def make_grad_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    except TypeError:
        return torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")


def _effective_batch_total(task_loaders, max_batches=None):
    natural_total = sum(len(loader) for loader in task_loaders.values())
    return min(natural_total, int(max_batches)) if max_batches else natural_total


def _format_duration(seconds: float) -> str:
    seconds = max(int(seconds), 0)
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"


def _gpu_memory_text() -> str:
    if DEVICE.type != "cuda":
        return "CPU"
    allocated = torch.cuda.memory_allocated() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    return f"GPU {allocated:.2f}/{peak:.2f} GB current/peak"


def _running_update(running: dict, key: str, value: float):
    record = running.setdefault(key, {"sum": 0.0, "count": 0})
    value = float(value)
    if np.isfinite(value):
        record["sum"] += value
        record["count"] += 1


def _running_summary(running: dict) -> dict:
    return {
        key: (float(value["sum"] / value["count"]) if value["count"] else float("nan"))
        for key, value in running.items()
    }


def _capture_gradients(model: nn.Module) -> dict:
    return {
        name: parameter.grad.detach().clone()
        for name, parameter in model.named_parameters()
        if parameter.grad is not None
    }


def _restore_gradients(model: nn.Module, saved_gradients: dict):
    parameters = dict(model.named_parameters())
    unknown = sorted(set(saved_gradients) - set(parameters))
    if unknown:
        raise RuntimeError("Checkpoint contains gradients for unknown parameters: " + ", ".join(unknown[:8]))
    for name, gradient in saved_gradients.items():
        parameter = parameters[name]
        if tuple(gradient.shape) != tuple(parameter.shape):
            raise RuntimeError(
                f"Gradient shape mismatch for {name}: {tuple(gradient.shape)} != {tuple(parameter.shape)}"
            )
        parameter.grad = gradient.to(device=parameter.device, dtype=parameter.dtype).clone()


def _train_progress_snapshot(model, batch_count, batches_consumed, samples_consumed,
                             active_tasks, round_index, running, elapsed_seconds) -> dict:
    remainder = int(batch_count % CFG.accumulation_steps)
    return {
        "batch_count": int(batch_count),
        "batches_consumed": {key: int(value) for key, value in batches_consumed.items()},
        "samples_consumed": {key: int(value) for key, value in samples_consumed.items()},
        # Keep the exact iterator set for the next interleaving round. A task that
        # yielded its final batch remains active until its next StopIteration, while
        # tasks exhausted in older rounds must stay removed; this distinction affects
        # the deterministic shuffled task order and gradient-accumulation windows.
        "active_tasks": list(active_tasks),
        "round_index": int(round_index),
        "elapsed_seconds": float(elapsed_seconds),
        "running_stats": {
            key: {"sum": float(value["sum"]), "count": int(value["count"])}
            for key, value in running.items()
        },
        "accumulation_remainder": remainder,
        # Gradients are scaled when AMP is active. The scaler state is checkpointed too,
        # so restoring both preserves an unfinished accumulation window exactly.
        "gradients": _capture_gradients(model) if remainder else {},
    }


def train_one_epoch(model, task_loaders, optimizer, scaler, ema, loss_fn, epoch,
                    max_batches=None, run_label="", run_seed=13,
                    resume_state=None, checkpoint_callback=None):
    model.train()
    tasks = sorted(task_loaders)
    resume_state = dict(resume_state or {})

    batch_count = int(resume_state.get("batch_count", 0))
    batches_consumed = {
        task: int(resume_state.get("batches_consumed", {}).get(task, 0))
        for task in tasks
    }
    samples_consumed = {
        task: int(resume_state.get("samples_consumed", {}).get(task, 0))
        for task in tasks
    }
    round_index = int(resume_state.get("round_index", 0))
    running = {
        key: {"sum": float(value["sum"]), "count": int(value["count"])}
        for key, value in resume_state.get("running_stats", {}).items()
    }
    elapsed_before = float(resume_state.get("elapsed_seconds", 0.0))
    total_batches = _effective_batch_total(task_loaders, max_batches)

    if batch_count != sum(batches_consumed.values()):
        raise RuntimeError(
            "Invalid mid-epoch checkpoint: global batch count does not equal the per-task counts."
        )
    if not 0 <= batch_count <= total_batches:
        raise RuntimeError(
            f"Invalid restored batch count {batch_count}; expected 0..{total_batches}."
        )
    for task in tasks:
        if not 0 <= batches_consumed[task] <= len(task_loaders[task]):
            raise RuntimeError(f"Invalid restored batch count for {task}: {batches_consumed[task]}")
        sampler_size = len(task_loaders[task].sampler)
        if not 0 <= samples_consumed[task] <= sampler_size:
            raise RuntimeError(f"Invalid restored sample position for {task}: {samples_consumed[task]}")

    expected_remainder = batch_count % CFG.accumulation_steps
    saved_remainder = int(resume_state.get("accumulation_remainder", expected_remainder))
    if saved_remainder != expected_remainder:
        raise RuntimeError("Invalid mid-epoch checkpoint: gradient-accumulation remainder differs.")
    saved_gradients = resume_state.get("gradients", {})
    if expected_remainder and not saved_gradients:
        raise RuntimeError("Mid-epoch checkpoint is missing unfinished accumulated gradients.")

    for task, loader in task_loaders.items():
        if not hasattr(loader.sampler, "set_epoch"):
            if samples_consumed[task]:
                raise RuntimeError(f"Sampler for {task} cannot restore a mid-epoch position.")
        else:
            loader.sampler.set_epoch(
                epoch,
                run_seed,
                start_sample=samples_consumed[task],
            )

    # Construct iterators only after assigning restore offsets. DataLoader creation can
    # consume a torch RNG value; restore the checkpoint RNG once more afterwards so
    # dropout and data augmentation continue from the exact saved state (workers=0).
    iterators = {task: iter(task_loaders[task]) for task in tasks}
    if resume_state.get("_checkpoint_rng_state"):
        _restore_rng_state(resume_state["_checkpoint_rng_state"])

    optimizer.zero_grad(set_to_none=True)
    if saved_gradients:
        _restore_gradients(model, saved_gradients)

    saved_active = resume_state.get("active_tasks")
    if saved_active is None:
        # Fresh runs enter with every task active. This fallback also permits an early
        # development-format checkpoint that predates active-task persistence.
        active = (
            list(tasks)
            if not resume_state
            else [task for task in tasks if batches_consumed[task] < len(task_loaders[task])]
        )
    else:
        active = list(saved_active)
        if len(active) != len(set(active)) or not set(active).issubset(tasks):
            raise RuntimeError(f"Invalid active task set in mid-epoch checkpoint: {active}")
        inactive = set(tasks) - set(active)
        not_exhausted = [
            task for task in inactive
            if samples_consumed[task] < len(task_loaders[task].sampler)
        ]
        if not_exhausted:
            raise RuntimeError(
                "Invalid mid-epoch checkpoint: inactive tasks are not exhausted: "
                + ", ".join(not_exhausted)
            )
    report_every = max(int(CFG.progress_every_batches), 1)
    checkpoint_batch_interval = max(int(CFG.checkpoint_every_batches), 0)
    checkpoint_seconds = max(float(CFG.checkpoint_every_minutes), 0.0) * 60.0
    last_checkpoint_batch = batch_count
    last_checkpoint_time = time.monotonic()
    started = time.perf_counter()

    if batch_count:
        print(
            f"[{run_label}] MID-EPOCH RESTORE: epoch {epoch + 1}/{EPOCHS}, "
            f"batch {batch_count:,}/{total_batches:,}; sampler offsets={samples_consumed}.",
            flush=True,
        )
    else:
        print(
            f"[{run_label}] epoch {epoch + 1}/{EPOCHS} TRAIN: {total_batches:,} batches; "
            "the first batch may pause while its NIfTI case is loaded and normalised.",
            flush=True,
        )

    progress = tqdm(
        total=total_batches,
        initial=batch_count,
        desc=f"{run_label} e{epoch + 1}/{EPOCHS} train",
        unit="batch",
        dynamic_ncols=True,
        mininterval=0.5,
    )

    try:
        stop = False
        while active and not stop:
            order = list(active)
            random.Random(run_seed * 1_000_003 + epoch * 10_007 + round_index).shuffle(order)
            still_active = set()

            for task in order:
                fetch_started = time.perf_counter()
                try:
                    batch = next(iterators[task])
                except StopIteration:
                    continue
                data_seconds = time.perf_counter() - fetch_started
                still_active.add(task)

                batch_samples = int(batch["image"].shape[0])
                x = batch["image"].to(DEVICE, non_blocking=True)
                y = batch["mask"].to(DEVICE, non_blocking=True)
                with autocast_context():
                    logits = model(x, task)
                    loss, _ = loss_fn(logits, y)
                    scaled_loss = loss / CFG.accumulation_steps
                scaler.scale(scaled_loss).backward()

                batch_count += 1
                batches_consumed[task] += 1
                samples_consumed[task] += batch_samples

                if batch_count % CFG.accumulation_steps == 0:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), CFG.gradient_clip)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    ema.update(model)

                loss_value = float(loss.detach())
                dice_value = mean_foreground_dice(logits.detach(), y)
                _running_update(running, f"{task}_loss", loss_value)
                _running_update(running, f"{task}_dice", dice_value)
                progress.update(1)
                progress.set_postfix(
                    task=task,
                    loss=f"{loss_value:.4f}",
                    data=f"{data_seconds:.2f}s",
                    refresh=False,
                )

                if batch_count == 1 or batch_count % report_every == 0 or batch_count == total_batches:
                    elapsed_attempt = time.perf_counter() - started
                    elapsed = elapsed_before + elapsed_attempt
                    newly_processed = max(batch_count - int(resume_state.get("batch_count", 0)), 1)
                    rate = newly_processed / max(elapsed_attempt, 1e-9)
                    eta = (total_batches - batch_count) / max(rate, 1e-9)
                    print(
                        f"[{run_label}] TRAIN {batch_count:,}/{total_batches:,} "
                        f"({100 * batch_count / max(total_batches, 1):.1f}%) | "
                        f"loss={loss_value:.4f} | data={data_seconds:.2f}s | "
                        f"elapsed={_format_duration(elapsed)} | ETA={_format_duration(eta)} | "
                        f"{_gpu_memory_text()}",
                        flush=True,
                    )

                if max_batches and batch_count >= max_batches:
                    stop = True
                    break

            active = [task for task in active if task in still_active]
            round_index += 1

            # Save only after a complete task round. This records an unambiguous next
            # task order and exact per-task sampler positions. Any unfinished gradient
            # accumulation is included in the checkpoint.
            if checkpoint_callback is not None and not stop and batch_count > last_checkpoint_batch:
                due_to_batches = (
                    checkpoint_batch_interval > 0
                    and batch_count - last_checkpoint_batch >= checkpoint_batch_interval
                )
                due_to_time = (
                    checkpoint_seconds > 0
                    and time.monotonic() - last_checkpoint_time >= checkpoint_seconds
                )
                if due_to_batches or due_to_time:
                    reason = "batch interval" if due_to_batches else "time interval"
                    checkpoint_callback(
                        _train_progress_snapshot(
                            model,
                            batch_count,
                            batches_consumed,
                            samples_consumed,
                            active,
                            round_index,
                            running,
                            elapsed_before + (time.perf_counter() - started),
                        ),
                        reason,
                    )
                    last_checkpoint_batch = batch_count
                    last_checkpoint_time = time.monotonic()
    finally:
        progress.close()

    if batch_count == 0:
        raise RuntimeError(f"{run_label}: no training batches were produced.")

    if batch_count % CFG.accumulation_steps:
        scaler.unscale_(optimizer)
        # Losses were divided by the nominal accumulation window. Correct the
        # shorter final window so smoke runs and naturally short epochs are not underweighted.
        final_window = batch_count % CFG.accumulation_steps
        correction = CFG.accumulation_steps / final_window
        for parameter in model.parameters():
            if parameter.grad is not None:
                parameter.grad.mul_(correction)
        nn.utils.clip_grad_norm_(model.parameters(), CFG.gradient_clip)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        ema.update(model)

    elapsed_total = elapsed_before + (time.perf_counter() - started)
    return _running_summary(running), elapsed_total


@torch.no_grad()
def validate_slices(model, task_loaders, loss_fn, max_batches=None, run_label="", epoch=0):
    model.eval()
    running = defaultdict(list)
    seen = 0
    total_batches = _effective_batch_total(task_loaders, max_batches)
    report_every = max(int(CFG.progress_every_batches), 1)
    started = time.perf_counter()

    print(f"[{run_label}] epoch {epoch + 1}/{EPOCHS} VALIDATE: {total_batches:,} batches.", flush=True)
    progress = tqdm(
        total=total_batches,
        desc=f"{run_label} e{epoch + 1}/{EPOCHS} val",
        unit="batch",
        dynamic_ncols=True,
        mininterval=0.5,
    )
    try:
        for task in sorted(task_loaders):
            for batch in task_loaders[task]:
                fetch_started = time.perf_counter()
                x = batch["image"].to(DEVICE, non_blocking=True)
                y = batch["mask"].to(DEVICE, non_blocking=True)
                data_seconds = time.perf_counter() - fetch_started
                with autocast_context():
                    logits = model(x, task)
                    loss, _ = loss_fn(logits, y)
                loss_value = float(loss)
                running[f"{task}_loss"].append(loss_value)
                running[f"{task}_dice"].append(mean_foreground_dice(logits, y))
                seen += 1
                progress.update(1)
                progress.set_postfix(task=task, loss=f"{loss_value:.4f}", refresh=False)

                if seen == 1 or seen % report_every == 0 or seen == total_batches:
                    elapsed = time.perf_counter() - started
                    rate = seen / max(elapsed, 1e-9)
                    eta = (total_batches - seen) / max(rate, 1e-9)
                    print(
                        f"[{run_label}] VAL {seen:,}/{total_batches:,} "
                        f"({100 * seen / max(total_batches, 1):.1f}%) | "
                        f"loss={loss_value:.4f} | elapsed={_format_duration(elapsed)} | "
                        f"ETA={_format_duration(eta)}",
                        flush=True,
                    )
                if max_batches and seen >= max_batches:
                    break
            if max_batches and seen >= max_batches:
                break
    finally:
        progress.close()

    if seen == 0:
        raise RuntimeError(f"{run_label}: no validation batches were produced.")
    summary = {k: float(np.nanmean(v)) for k, v in running.items()}
    dice_values = [v for k, v in summary.items() if k.endswith("_dice")]
    summary["macro_task_dice"] = float(np.nanmean(dice_values))
    return summary


def checkpoint_metadata_reasons(checkpoint: dict, experiment_name: str, seed: int,
                                kwargs: dict, require_complete: bool = True) -> List[str]:
    selected_tasks = sorted(kwargs.get("selected_tasks") or TASK_SPECS)
    expected_fingerprint = fingerprint_for_tasks(selected_tasks)
    reasons = []
    if require_complete and checkpoint.get("training_complete") is not True:
        reasons.append("training was not completed")
    if checkpoint.get("pipeline_revision") != PIPELINE_REVISION:
        reasons.append("pipeline revision differs")
    if checkpoint.get("dataset_fingerprint") != expected_fingerprint:
        reasons.append("dataset fingerprint differs")
    if checkpoint.get("experiment") != experiment_name:
        reasons.append("experiment name differs")
    if int(checkpoint.get("seed", -1)) != int(seed):
        reasons.append("seed differs")
    if sorted(checkpoint.get("task_specs", {})) != selected_tasks:
        reasons.append("selected task set differs")
    for key in ("adaptive_slice", "use_film", "use_boundary"):
        if checkpoint.get(key) != kwargs.get(key):
            reasons.append(f"{key} differs")
    old_config = checkpoint.get("config", {})
    current_config = asdict(CFG)
    changed_config = [
        key for key in TRAINING_CONFIG_KEYS
        if old_config.get(key) != current_config.get(key)
    ]
    if changed_config:
        reasons.append("training config differs: " + ", ".join(changed_config))
    return reasons


def _atomic_torch_save(payload: dict, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    try:
        with open(temporary, "wb") as handle:
            torch.save(payload, handle)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    except Exception:
        temporary.unlink(missing_ok=True)
        raise


def _atomic_json_save(payload: dict, path: Path):
    temporary = path.with_suffix(path.suffix + ".tmp")
    try:
        with open(temporary, "w", encoding="utf-8") as handle:
            json.dump(payload, handle, indent=2)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    except Exception:
        temporary.unlink(missing_ok=True)
        raise


def _write_history(history: List[dict], path: Path):
    temporary = path.with_suffix(path.suffix + ".tmp")
    pd.DataFrame(history).to_csv(temporary, index=False)
    os.replace(temporary, path)


def _rng_state() -> dict:
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def _restore_rng_state(state: Optional[dict]):
    if not state:
        return
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"].cpu())
    if torch.cuda.is_available() and state.get("cuda") is not None:
        torch.cuda.set_rng_state_all([item.cpu() for item in state["cuda"]])


def _first_nonfinite(value, path="checkpoint") -> Optional[str]:
    if torch.is_tensor(value):
        if (value.is_floating_point() or value.is_complex()) and not bool(torch.isfinite(value).all()):
            return path
        return None
    if isinstance(value, dict):
        for key, item in value.items():
            found = _first_nonfinite(item, f"{path}.{key}")
            if found:
                return found
    elif isinstance(value, (list, tuple)):
        for index, item in enumerate(value):
            found = _first_nonfinite(item, f"{path}[{index}]")
            if found:
                return found
    return None


def _state_dict_layout_reasons(candidate: dict, reference: dict, label: str) -> List[str]:
    reasons = []
    missing = sorted(set(reference) - set(candidate))
    unexpected = sorted(set(candidate) - set(reference))
    if missing:
        reasons.append(f"{label} is missing {len(missing)} tensor(s): {', '.join(missing[:5])}")
    if unexpected:
        reasons.append(f"{label} has {len(unexpected)} unexpected tensor(s): {', '.join(unexpected[:5])}")
    for key in set(candidate) & set(reference):
        if tuple(candidate[key].shape) != tuple(reference[key].shape):
            reasons.append(
                f"{label}.{key} shape differs: {tuple(candidate[key].shape)} != {tuple(reference[key].shape)}"
            )
            if len(reasons) >= 10:
                break
    return reasons


def _train_progress_reasons(progress_state: dict, task_loaders: dict) -> List[str]:
    reasons = []
    tasks = sorted(task_loaders)
    try:
        batch_count = int(progress_state["batch_count"])
        batches = {task: int(progress_state["batches_consumed"].get(task, 0)) for task in tasks}
        samples = {task: int(progress_state["samples_consumed"].get(task, 0)) for task in tasks}
        active = progress_state.get("active_tasks")
        if batch_count != sum(batches.values()):
            reasons.append("mid-epoch global/per-task batch counts disagree")
        configured_limit = (
            CFG.full_train_batches_per_epoch if CFG.run_full_paper
            else CFG.max_smoke_train_batches
        )
        effective_limit = _effective_batch_total(task_loaders, configured_limit)
        if not 0 <= batch_count <= effective_limit:
            reasons.append(
                f"saved global batch position {batch_count} exceeds the effective epoch limit {effective_limit}"
            )
        for task in tasks:
            if not 0 <= batches[task] <= len(task_loaders[task]):
                reasons.append(f"invalid saved batch position for {task}")
            if not 0 <= samples[task] <= len(task_loaders[task].sampler):
                reasons.append(f"invalid saved sample position for {task}")
            expected_batches = math.ceil(samples[task] / CFG.batch_size) if samples[task] else 0
            if batches[task] != expected_batches:
                reasons.append(
                    f"saved sample/batch positions disagree for {task}: "
                    f"samples={samples[task]}, batches={batches[task]}"
                )
        if not isinstance(active, list):
            reasons.append("mid-epoch active task list is missing")
        elif len(active) != len(set(active)) or not set(active).issubset(tasks):
            reasons.append("mid-epoch active task list is invalid")
        else:
            inactive = set(tasks) - set(active)
            for task in inactive:
                if samples[task] < len(task_loaders[task].sampler):
                    reasons.append(f"inactive task {task} is not exhausted")
        remainder = batch_count % CFG.accumulation_steps
        if int(progress_state.get("accumulation_remainder", remainder)) != remainder:
            reasons.append("gradient-accumulation remainder disagrees with batch count")
        if remainder and not progress_state.get("gradients"):
            reasons.append("unfinished gradient accumulation is missing its gradients")
    except Exception as exc:
        reasons.append(f"invalid mid-epoch progress structure: {exc}")
    return reasons


def _quarantine_resume_checkpoint(path: Path, reason: str) -> Optional[Path]:
    if not path.exists():
        return None
    quarantine = OUTPUT_ROOT / "checkpoints_quarantine"
    quarantine.mkdir(parents=True, exist_ok=True)
    stamp = time.strftime("%Y%m%d_%H%M%S", time.localtime())
    destination = quarantine / f"{path.stem}_{stamp}{path.suffix}"
    counter = 1
    while destination.exists():
        destination = quarantine / f"{path.stem}_{stamp}_{counter}{path.suffix}"
        counter += 1
    path.replace(destination)
    destination.with_suffix(destination.suffix + ".reason.txt").write_text(reason, encoding="utf-8")
    return destination


def run_training(experiment_name: str, seed: int, task_specs: dict, task_splits: dict,
                 adaptive_slice=True, use_film=True, use_boundary=True, selected_tasks=None):
    seed_everything(seed)
    selected_tasks = sorted(selected_tasks or task_specs)
    specs = {task: task_specs[task] for task in selected_tasks}
    exp_loaders = {task: loaders[task] for task in selected_tasks}
    train_loaders = {task: exp_loaders[task]["train"] for task in selected_tasks}
    val_loaders = {task: exp_loaders[task]["val"] for task in selected_tasks}
    training_fingerprint = fingerprint_for_tasks(selected_tasks)
    kwargs = {
        "adaptive_slice": adaptive_slice,
        "use_film": use_film,
        "use_boundary": use_boundary,
        "selected_tasks": selected_tasks,
    }
    run_label = f"{experiment_name}/seed{seed}"

    net = UniGDNetR(
        specs,
        CFG.base_channels,
        CFG.task_embedding_dim,
        CFG.dropout,
        adaptive_slice=adaptive_slice,
        use_film=use_film,
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(net.parameters(), lr=CFG.learning_rate, weight_decay=CFG.weight_decay)
    scheduler = make_scheduler(optimizer, EPOCHS)
    scaler = make_grad_scaler()
    ema = EMA(net, CFG.ema_decay)
    loss_fn = CompoundSegmentationLoss(use_boundary=use_boundary)

    history, best, bad_epochs = [], -np.inf, 0
    start_epoch = 0
    resume_phase = "train"
    resume_train_state = None
    pending_train_log = None
    pending_train_seconds = None
    ckpt_path = OUTPUT_ROOT / "checkpoints" / f"{experiment_name}_seed{seed}.pt"
    resume_path = OUTPUT_ROOT / "checkpoints" / f"{experiment_name}_seed{seed}_resume.pt"
    resume_status_path = OUTPUT_ROOT / "checkpoints" / f"{experiment_name}_seed{seed}_resume_status.json"
    history_path = OUTPUT_ROOT / "logs" / f"{experiment_name}_seed{seed}_history.csv"
    max_train = None if CFG.run_full_paper else CFG.max_smoke_train_batches
    max_val = None if CFG.run_full_paper else CFG.max_smoke_val_batches
    stopped_early = False

    train_counts = {task: len(train_loaders[task]) for task in selected_tasks}
    val_counts = {task: len(val_loaders[task]) for task in selected_tasks}
    parameter_count = sum(parameter.numel() for parameter in net.parameters())
    print("\n" + "=" * 88, flush=True)
    print(f"START {run_label} | tasks={selected_tasks} | device={DEVICE}", flush=True)
    print(f"Parameters: {parameter_count:,} | epochs: {EPOCHS}", flush=True)
    print(f"Training batches by task: {train_counts}", flush=True)
    print(f"Validation batches by task: {val_counts}", flush=True)
    print(
        f"Effective batches/epoch: train={_effective_batch_total(train_loaders, max_train):,}, "
        f"validation={_effective_batch_total(val_loaders, max_val):,}",
        flush=True,
    )
    print(
        f"Recovery checkpoint: every {CFG.checkpoint_every_batches:,} batches or "
        f"{CFG.checkpoint_every_minutes:g} minutes -> {resume_path.name}",
        flush=True,
    )
    if CFG.num_workers != 0:
        print(
            "WARNING: exact augmentation-RNG replay is guaranteed with num_workers=0; "
            "sampler positions and optimisation state will still be restored.",
            flush=True,
        )

    if resume_path.exists():
        print(f"Checking recovery checkpoint: {resume_path}", flush=True)
        try:
            resume = torch_load_compat(resume_path, map_location=DEVICE)
            reasons = checkpoint_metadata_reasons(
                resume, experiment_name, seed, kwargs, require_complete=False
            )
            required = {"model", "ema", "optimizer", "scheduler", "scaler", "epoch_completed"}
            missing = sorted(required - set(resume))
            if missing:
                reasons.append("resume state missing: " + ", ".join(missing))
            if "model" in resume:
                reasons.extend(_state_dict_layout_reasons(resume["model"], net.state_dict(), "model"))
                bad_tensor = _first_nonfinite(resume["model"], "model")
                if bad_tensor:
                    reasons.append(f"non-finite values detected in {bad_tensor}")
            if "ema" in resume:
                reasons.extend(_state_dict_layout_reasons(resume["ema"], net.state_dict(), "EMA"))
                bad_tensor = _first_nonfinite(resume["ema"], "EMA")
                if bad_tensor:
                    reasons.append(f"non-finite values detected in {bad_tensor}")
            for state_name in ("optimizer", "scaler"):
                if state_name in resume:
                    bad_tensor = _first_nonfinite(resume[state_name], state_name)
                    if bad_tensor:
                        reasons.append(f"non-finite values detected in {bad_tensor}")

            format_version = int(resume.get("resume_format_version", 1))
            if format_version not in {1, 2}:
                reasons.append(f"unsupported resume format version: {format_version}")
            saved_engine_revision = resume.get("resume_engine_revision")
            if saved_engine_revision is not None and saved_engine_revision != RESUME_ENGINE_REVISION:
                reasons.append(
                    "resume engine revision differs: "
                    f"{saved_engine_revision!r} != {RESUME_ENGINE_REVISION!r}"
                )
            candidate_phase = resume.get("resume_phase", "epoch_boundary")
            candidate_epoch = int(resume.get("epoch_index", resume.get("epoch_completed", 0)))
            saved_batch = int((resume.get("train_progress") or {}).get("batch_count", 0))
            print(
                f"Recovery header: format={format_version}, phase={candidate_phase}, "
                f"epoch_index={candidate_epoch}, batch={saved_batch:,}, "
                f"saved={resume.get('saved_utc', 'unknown')}",
                flush=True,
            )
            if not 0 <= candidate_epoch <= EPOCHS:
                reasons.append(f"invalid resume epoch index: {candidate_epoch}")
            history_candidate = resume.get("history", [])
            if not isinstance(history_candidate, list):
                reasons.append("resume history is not a list")
            elif len(history_candidate) != candidate_epoch:
                reasons.append(
                    f"resume history length {len(history_candidate)} disagrees with epoch index {candidate_epoch}"
                )
            if format_version >= 2 and not isinstance(resume.get("rng_state"), dict):
                reasons.append("version-2 resume is missing its RNG state")
            if format_version >= 2 and candidate_phase not in {"train", "validation"}:
                reasons.append(f"unknown resume phase: {candidate_phase}")
            if candidate_phase == "validation" and candidate_epoch >= EPOCHS:
                reasons.append("validation-phase resume points beyond the final trainable epoch")
            if candidate_phase == "train" and resume.get("train_progress"):
                reasons.extend(_train_progress_reasons(resume["train_progress"], train_loaders))
                bad_tensor = _first_nonfinite(resume["train_progress"], "train_progress")
                if bad_tensor:
                    reasons.append(f"non-finite values detected in {bad_tensor}")
            if candidate_phase == "validation":
                if not isinstance(resume.get("pending_train_log"), dict):
                    reasons.append("validation-phase resume is missing the completed training log")
                if not isinstance(resume.get("pending_train_seconds"), (int, float)):
                    reasons.append("validation-phase resume is missing its accumulated training time")

            if reasons:
                reason = "; ".join(reasons)
                moved_to = _quarantine_resume_checkpoint(resume_path, reason)
                print(f"Rejected and quarantined unsafe recovery checkpoint: {moved_to}", flush=True)
                print("Reason:", reason, flush=True)
            else:
                # Strict layout checks above precede every state mutation.
                net.load_state_dict(resume["model"], strict=True)
                ema.shadow = {key: value.to(DEVICE) for key, value in resume["ema"].items()}
                optimizer.load_state_dict(resume["optimizer"])
                scheduler.load_state_dict(resume["scheduler"])
                scaler.load_state_dict(resume["scaler"])
                history = list(resume.get("history", []))
                best = float(resume.get("best", -np.inf))
                bad_epochs = int(resume.get("bad_epochs", 0))
                _restore_rng_state(resume.get("rng_state"))

                if format_version == 1 or candidate_phase == "epoch_boundary":
                    # Backward compatibility with the notebook's existing epoch-only file.
                    start_epoch = int(resume["epoch_completed"])
                    if format_version == 1:
                        print(
                            "LEGACY CHECKPOINT: this file stores only a completed epoch, not a "
                            "mid-epoch batch. It will resume safely at the next epoch and will be "
                            "upgraded to a batch-level checkpoint at the next periodic save.",
                            flush=True,
                        )
                    resume_phase = "train"
                    resume_train_state = None
                    print(
                        f"RESUMED {run_label} after completed epoch {start_epoch}; "
                        f"best validation Dice={best:.5f}.",
                        flush=True,
                    )
                elif candidate_phase == "train":
                    start_epoch = candidate_epoch
                    resume_phase = "train"
                    resume_train_state = dict(resume.get("train_progress") or {}) or None
                    if resume_train_state is not None:
                        resume_train_state["_checkpoint_rng_state"] = resume.get("rng_state")
                        saved_batch = int(resume_train_state.get("batch_count", 0))
                        print(
                            f"RESUMED {run_label} inside epoch {start_epoch + 1} at "
                            f"batch {saved_batch:,}; completed samples will not be reloaded.",
                            flush=True,
                        )
                    else:
                        print(f"RESUMED {run_label} at the start of epoch {start_epoch + 1}.", flush=True)
                else:
                    start_epoch = candidate_epoch
                    resume_phase = "validation"
                    pending_train_log = dict(resume["pending_train_log"])
                    pending_train_seconds = float(resume["pending_train_seconds"])
                    print(
                        f"RESUMED {run_label} after training epoch {start_epoch + 1}; "
                        "training will not repeat and validation will restart.",
                        flush=True,
                    )
        except Exception as exc:
            reason = f"unreadable or invalid recovery checkpoint: {type(exc).__name__}: {exc}"
            moved_to = _quarantine_resume_checkpoint(resume_path, reason)
            print(f"Rejected and quarantined recovery checkpoint: {moved_to}", flush=True)
            print("Reason:", reason, flush=True)

    def save_resume(phase: str, epoch_index: int, epoch_completed: int,
                    train_progress=None, completed_train_log=None,
                    completed_train_seconds=None, reason="periodic"):
        payload = {
            "resume_format_version": 2,
            "resume_engine_revision": RESUME_ENGINE_REVISION,
            "resume_phase": phase,
            "epoch_index": int(epoch_index),
            "epoch_completed": int(epoch_completed),
            "train_progress": train_progress,
            "pending_train_log": completed_train_log,
            "pending_train_seconds": completed_train_seconds,
            "model": net.state_dict(),
            "ema": ema.shadow,
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "history": history,
            "best": best,
            "bad_epochs": bad_epochs,
            "rng_state": _rng_state(),
            "task_specs": specs,
            "seed": seed,
            "experiment": experiment_name,
            "config": asdict(CFG),
            "adaptive_slice": adaptive_slice,
            "use_film": use_film,
            "use_boundary": use_boundary,
            "dataset_fingerprint": training_fingerprint,
            "task_dataset_fingerprints": {
                task: TASK_DATASET_FINGERPRINTS[task] for task in selected_tasks
            },
            "pipeline_revision": PIPELINE_REVISION,
            "training_complete": False,
            "saved_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "save_reason": reason,
        }
        _atomic_torch_save(payload, resume_path)

        batch_count = int((train_progress or {}).get("batch_count", 0))
        status = {
            "experiment": experiment_name,
            "seed": int(seed),
            "phase": phase,
            "epoch_human": int(epoch_index + 1) if epoch_index < EPOCHS else int(EPOCHS),
            "epochs_total": int(EPOCHS),
            "batch": batch_count,
            "batches_total": int(_effective_batch_total(train_loaders, max_train)),
            "batches_consumed_by_task": (train_progress or {}).get("batches_consumed", {}),
            "saved_utc": payload["saved_utc"],
            "reason": reason,
            "checkpoint": str(resume_path),
        }
        try:
            _atomic_json_save(status, resume_status_path)
        except Exception as exc:
            print(f"WARNING: checkpoint saved, but status JSON could not be written: {exc}", flush=True)
        print(
            f"[{run_label}] RECOVERY CHECKPOINT SAVED | phase={phase} | "
            f"epoch={status['epoch_human']}/{EPOCHS} | batch={batch_count:,} | "
            f"reason={reason} | {resume_path}",
            flush=True,
        )

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    try:
        for epoch in range(start_epoch, EPOCHS):
            epoch_started = time.time()
            print(f"\n[{run_label}] BEGIN EPOCH {epoch + 1}/{EPOCHS}", flush=True)

            if epoch == start_epoch and resume_phase == "validation":
                train_log = pending_train_log
                train_seconds = float(pending_train_seconds)
                print(
                    f"[{run_label}] Training for epoch {epoch + 1} was already checkpointed; "
                    "continuing with validation.",
                    flush=True,
                )
            else:
                state_for_epoch = resume_train_state if epoch == start_epoch else None

                def periodic_checkpoint(progress_state, reason):
                    save_resume(
                        phase="train",
                        epoch_index=epoch,
                        epoch_completed=epoch,
                        train_progress=progress_state,
                        completed_train_log=None,
                        completed_train_seconds=None,
                        reason=reason,
                    )

                train_log, train_seconds = train_one_epoch(
                    net,
                    train_loaders,
                    optimizer,
                    scaler,
                    ema,
                    loss_fn,
                    epoch,
                    max_batches=max_train,
                    run_label=run_label,
                    run_seed=seed,
                    resume_state=state_for_epoch,
                    checkpoint_callback=periodic_checkpoint,
                )

                # Protect the full 10–15-hour training phase before validation begins.
                save_resume(
                    phase="validation",
                    epoch_index=epoch,
                    epoch_completed=epoch,
                    train_progress=None,
                    completed_train_log=train_log,
                    completed_train_seconds=train_seconds,
                    reason="training phase completed",
                )

            resume_train_state = None
            resume_phase = "train"
            pending_train_log = None
            pending_train_seconds = None

            validation_started = time.time()
            eval_net = UniGDNetR(
                specs,
                CFG.base_channels,
                CFG.task_embedding_dim,
                CFG.dropout,
                adaptive_slice=adaptive_slice,
                use_film=use_film,
            ).to(DEVICE)
            ema.copy_to(eval_net)
            val_log = validate_slices(
                eval_net,
                val_loaders,
                loss_fn,
                max_batches=max_val,
                run_label=run_label,
                epoch=epoch,
            )
            scheduler.step()
            row = {
                "epoch": epoch + 1,
                "seconds": train_seconds + (time.time() - validation_started),
                "lr": optimizer.param_groups[0]["lr"],
                **{f"train_{key}": value for key, value in train_log.items()},
                **{f"val_{key}": value for key, value in val_log.items()},
            }
            history.append(row)
            _write_history(history, history_path)

            score = val_log["macro_task_dice"]
            if not np.isfinite(score):
                raise RuntimeError(
                    f"Non-finite validation score in {experiment_name}, seed {seed}, epoch {epoch + 1}."
                )

            improved = score > best
            if improved:
                best, bad_epochs = score, 0
                _atomic_torch_save({
                    "model": ema.shadow,
                    "task_specs": specs,
                    "seed": seed,
                    "experiment": experiment_name,
                    "config": asdict(CFG),
                    "best_val_macro_task_dice": best,
                    "adaptive_slice": adaptive_slice,
                    "use_film": use_film,
                    "use_boundary": use_boundary,
                    "dataset_fingerprint": training_fingerprint,
                    "task_dataset_fingerprints": {
                        task: TASK_DATASET_FINGERPRINTS[task] for task in selected_tasks
                    },
                    "pipeline_revision": PIPELINE_REVISION,
                    "best_epoch": epoch + 1,
                    "training_complete": False,
                }, ckpt_path)
            else:
                bad_epochs += 1

            should_stop = CFG.run_full_paper and bad_epochs >= CFG.patience
            # The boundary checkpoint lets old and new versions resume after this epoch.
            save_resume(
                phase="train",
                epoch_index=epoch + 1,
                epoch_completed=epoch + 1,
                train_progress=None,
                completed_train_log=None,
                completed_train_seconds=None,
                reason="epoch completed",
            )

            print(
                f"[{run_label}] END EPOCH {epoch + 1}/{EPOCHS} | "
                f"val_macro_dice={score:.5f} | best={best:.5f} | "
                f"improved={improved} | bad_epochs={bad_epochs} | "
                f"time={_format_duration(row['seconds'])} | {_gpu_memory_text()}",
                flush=True,
            )

            del eval_net
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            if should_stop:
                print(f"[{run_label}] Early stopping after {bad_epochs} non-improving epochs.", flush=True)
                stopped_early = True
                break
    except KeyboardInterrupt:
        print(
            f"\n[{run_label}] Training interrupted. The last atomic recovery checkpoint remains at:\n"
            f"  {resume_path}\nRerun the training cell to continue from that saved position.",
            flush=True,
        )
        raise

    if not ckpt_path.exists():
        raise RuntimeError(f"Training ended without producing a checkpoint: {ckpt_path}")

    completed_checkpoint = torch_load_compat(ckpt_path, map_location="cpu")
    completed_checkpoint.update({
        "training_complete": True,
        "epochs_completed": len(history),
        "stopped_early": stopped_early,
        "history_file": str(history_path),
    })
    _atomic_torch_save(completed_checkpoint, ckpt_path)
    resume_path.unlink(missing_ok=True)
    resume_status_path.unlink(missing_ok=True)
    print(f"COMPLETE {run_label} -> {ckpt_path}", flush=True)
    return ckpt_path, history

## 11. Volumetric inference and paper-grade metrics

Predictions are resized back to the original in-plane resolution and reassembled into 3D volumes.
Positive-only Dice is reported alongside all-case Dice to prevent empty-slice inflation.


In [ ]:
def prepare_inference_slice(volume: np.ndarray, z: int, k: int, size: int) -> torch.Tensor:
    radius = k // 2
    indices = [min(max(z+d, 0), volume.shape[-1]-1) for d in range(-radius, radius+1)]
    arr = np.stack([volume[m,:,:,zi] for m in range(volume.shape[0]) for zi in indices], axis=0)
    x = torch.from_numpy(arr.astype(np.float32))[None]
    return F.interpolate(x, size=(size,size), mode="bilinear", align_corners=False)[0]

def apply_corruption(x: torch.Tensor, name: str, severity: float) -> torch.Tensor:
    if name == "clean":
        return x
    if name == "gaussian_noise":
        return x + torch.randn_like(x) * severity
    if name == "blur":
        kernel = int(severity)
        kernel += (kernel + 1) % 2
        return F.avg_pool2d(x, kernel, stride=1, padding=kernel//2)
    if name == "resolution":
        h, w = x.shape[-2:]
        small = F.interpolate(x, scale_factor=severity, mode="bilinear", align_corners=False)
        return F.interpolate(small, size=(h,w), mode="bilinear", align_corners=False)
    if name == "gamma":
        sign = torch.sign(x)
        return sign * torch.abs(x).clamp_min(1e-6).pow(severity)
    raise KeyError(name)

def set_mc_dropout(model: nn.Module, enabled: bool):
    model.eval()
    if enabled:
        for module in model.modules():
            if isinstance(module, (nn.Dropout, nn.Dropout2d, nn.Dropout3d)):
                module.train()

@torch.no_grad()
def predict_volume(model, record: dict, temperature: float = 1.0, mc_samples: int = 1,
                   missing_modalities: Optional[List[int]] = None,
                   corruption: Tuple[str,float] = ("clean", 0.0)):
    task = record["task"]
    volume, mask = load_case(record)
    h, w, depth = mask.shape
    probs_out, entropy_out = [], []
    batch_x = []
    for z in range(depth):
        x = prepare_inference_slice(volume, z, CFG.context_slices, CFG.image_size)
        if missing_modalities:
            for m in missing_modalities:
                x[m*CFG.context_slices:(m+1)*CFG.context_slices] = 0
        x = apply_corruption(x[None], *corruption)[0]
        batch_x.append(x)
        if len(batch_x) == CFG.batch_size or z == depth-1:
            xb = torch.stack(batch_x).to(DEVICE)
            sample_probs = []
            set_mc_dropout(model, mc_samples > 1)
            for _ in range(mc_samples):
                logits = model(xb, task) / max(float(temperature), 1e-3)
                sample_probs.append(torch.softmax(logits, dim=1))
            mean_prob = torch.stack(sample_probs).mean(0)
            entropy = -(mean_prob.clamp_min(1e-8) * mean_prob.clamp_min(1e-8).log()).sum(1)
            mean_prob = F.interpolate(mean_prob, size=(h,w), mode="bilinear", align_corners=False)
            entropy = F.interpolate(entropy[:,None], size=(h,w), mode="bilinear", align_corners=False)[:,0]
            probs_out.extend(mean_prob.cpu().numpy())
            entropy_out.extend(entropy.cpu().numpy())
            batch_x = []
    set_mc_dropout(model, False)
    probs = np.stack(probs_out, axis=-1)  # classes,H,W,D
    entropy = np.stack(entropy_out, axis=-1)
    pred = probs.argmax(0).astype(np.int16)
    return pred, probs, entropy, mask

def binary_metrics(pred: np.ndarray, gt: np.ndarray) -> dict:
    pred, gt = pred.astype(bool), gt.astype(bool)
    tp = np.logical_and(pred, gt).sum()
    fp = np.logical_and(pred, ~gt).sum()
    fn = np.logical_and(~pred, gt).sum()
    denom = pred.sum() + gt.sum()
    dice = (2*tp/denom) if denom else np.nan
    iou = tp/(tp+fp+fn) if (tp+fp+fn) else np.nan
    sensitivity = tp/(tp+fn) if (tp+fn) else np.nan
    precision = tp/(tp+fp) if (tp+fp) else np.nan
    return {"dice": dice, "iou": iou, "sensitivity": sensitivity, "precision": precision,
            "gt_voxels": int(gt.sum()), "pred_voxels": int(pred.sum())}

def hd95_binary(pred: np.ndarray, gt: np.ndarray, spacing=(1.0,1.0,1.0)) -> float:
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not pred.any() or not gt.any():
        return np.nan
    pred_border = pred ^ ndi.binary_erosion(pred)
    gt_border = gt ^ ndi.binary_erosion(gt)
    d_gt = ndi.distance_transform_edt(~gt_border, sampling=spacing)
    d_pred = ndi.distance_transform_edt(~pred_border, sampling=spacing)
    distances = np.concatenate([d_gt[pred_border], d_pred[gt_border]])
    return float(np.percentile(distances, 95)) if len(distances) else 0.0

def ece_score(probs: np.ndarray, gt: np.ndarray, bins: int = 15, max_voxels: int = 250000) -> float:
    conf = probs.max(0).ravel()
    pred = probs.argmax(0).ravel()
    truth = gt.ravel()
    if len(conf) > max_voxels:
        rng = np.random.default_rng(13)
        idx = rng.choice(len(conf), max_voxels, replace=False)
        conf, pred, truth = conf[idx], pred[idx], truth[idx]
    edges = np.linspace(0, 1, bins+1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs((pred[m] == truth[m]).mean() - conf[m].mean())
    return float(ece)

def task_regions(task: str, pred: np.ndarray, gt: np.ndarray):
    if task == "brats":
        return {
            "WT": (np.isin(pred, [1, 2, 3]), np.isin(gt, [1, 2, 3])),
            "TC": (np.isin(pred, [1, 3]), np.isin(gt, [1, 3])),
            "NETC": (pred == 1, gt == 1),
            "SNFH": (pred == 2, gt == 2),
            "ET": (pred == 3, gt == 3),
            "RC": (pred == 4, gt == 4),
        }
    names = TASK_SPECS[task]["class_names"]
    return {names[c]: (pred==c, gt==c) for c in range(1, TASK_SPECS[task]["classes"])}

def evaluate_records(model, records: List[dict], task: str, experiment: str, seed: int,
                     temperature: float = 1.0, mc_samples: int = 1,
                     missing_modalities=None, corruption=("clean",0.0)) -> pd.DataFrame:
    rows = []
    for record in tqdm(records, desc=f"Evaluate {experiment}/{task}"):
        pred, probs, entropy, gt = predict_volume(model, record, temperature, mc_samples,
                                                  missing_modalities, corruption)
        affine = nib.as_closest_canonical(nib.load(record["images"][0])).affine
        spacing = tuple(np.sqrt((affine[:3,:3]**2).sum(0)).tolist())
        case_ece = ece_score(probs, gt)
        for region, (p, y) in task_regions(task, pred, gt).items():
            metrics = binary_metrics(p, y)
            metrics["hd95"] = hd95_binary(p, y, spacing)
            rows.append({
                "experiment": experiment, "seed": seed, "task": task,
                "case_id": record["case_id"],
                "patient_id": record.get("patient_id", record.get("group_id", record["case_id"])),
                "region": region,
                "ece": case_ece, "mean_entropy": float(entropy.mean()),
                "missing_modalities": str(missing_modalities or []),
                "corruption": corruption[0], "severity": corruption[1], **metrics,
            })
    return pd.DataFrame(rows)

## 12. Post-hoc temperature calibration

Temperature is fitted on validation pixels only and then frozen before test evaluation.


In [ ]:
def fit_temperature(model, loader, task: str, max_batches: int = 80, sampling_seed: int = 13) -> float:
    generator = torch.Generator(device=DEVICE)
    generator.manual_seed(int(sampling_seed))
    model.eval()
    logits_list, labels_list = [], []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            x = batch["image"].to(DEVICE)
            y = batch["mask"].to(DEVICE)
            logits = model(x, task).float()
            # Foreground/background-balanced spatial subsample to avoid background-dominated calibration.
            flat_logits = logits.permute(0,2,3,1).reshape(-1, logits.shape[1])
            flat_y = y.reshape(-1)
            fg = torch.where(flat_y > 0)[0]
            bg = torch.where(flat_y == 0)[0]
            half = 10000
            fg_idx = fg[torch.randperm(len(fg), generator=generator, device=DEVICE)[:min(half, len(fg))]] if len(fg) else fg
            bg_idx = bg[torch.randperm(len(bg), generator=generator, device=DEVICE)[:min(half, len(bg))]] if len(bg) else bg
            idx = torch.cat([fg_idx, bg_idx])
            if len(idx) == 0:
                continue
            logits_list.append(flat_logits[idx].cpu())
            labels_list.append(flat_y[idx].cpu())
            if i+1 >= max_batches:
                break
    if not logits_list:
        raise RuntimeError(f"Temperature calibration produced no usable voxels for task {task}.")
    logits = torch.cat(logits_list).to(DEVICE)
    labels = torch.cat(labels_list).to(DEVICE)
    log_t = torch.zeros(1, device=DEVICE, requires_grad=True)
    optimizer = torch.optim.LBFGS([log_t], lr=0.05, max_iter=60, line_search_fn="strong_wolfe")
    def closure():
        optimizer.zero_grad()
        t = log_t.exp().clamp(0.05, 10.0)
        loss = F.cross_entropy(logits/t, labels)
        loss.backward()
        return loss
    optimizer.step(closure)
    return float(log_t.exp().clamp(0.05,10.0).detach())

def load_checkpoint(path: Path):
    ckpt = torch_load_compat(path, map_location=DEVICE)
    if not ckpt.get("training_complete", False):
        raise RuntimeError(f"Refusing incomplete checkpoint: {path}")
    net = UniGDNetR(
        ckpt["task_specs"], CFG.base_channels, CFG.task_embedding_dim, CFG.dropout,
        adaptive_slice=ckpt["adaptive_slice"], use_film=ckpt["use_film"]
    ).to(DEVICE)
    net.load_state_dict(ckpt["model"], strict=True)
    net.eval()
    return net, ckpt

## 13. Experiment registry

For a paper, run the main model and all ablations with all three seeds. Single-task GDNet controls
whether sharing actually helps. `Shared-no-FiLM` tests task conditioning. `Uniform-slice` tests the
reliability gate. `No-boundary` tests the boundary term.


In [ ]:
EXPERIMENTS = {
    "UniGDNetR": dict(adaptive_slice=True, use_film=True, use_boundary=True, selected_tasks=None),
    "Uniform-slice": dict(adaptive_slice=False, use_film=True, use_boundary=True, selected_tasks=None),
    "Shared-no-FiLM": dict(adaptive_slice=True, use_film=False, use_boundary=True, selected_tasks=None),
    "No-boundary": dict(adaptive_slice=True, use_film=True, use_boundary=False, selected_tasks=None),
}
for task in TASK_SPECS:
    EXPERIMENTS[f"SingleTask-{task}"] = dict(
        adaptive_slice=True, use_film=True, use_boundary=True, selected_tasks=[task]
    )

display(pd.DataFrame(EXPERIMENTS).T)


In [ ]:
# ========================= FULL AUTOMATIC MODE =========================
# Run All requires no editing. The complete paper matrix is mandatory in
# full-paper mode; smoke mode evaluates only its configured main-model run.
EVALUATION_PHASE = "all" if CFG.run_full_paper else "main"
EVALUATION_SEEDS = list(SEEDS)
REQUIRE_ALL_CHECKPOINTS = True
AUTO_EVALUATION = True
# ======================================================================

print("FULL AUTOMATIC EVALUATION ENABLED", flush=True)
print("Phase:", EVALUATION_PHASE, "| seeds:", EVALUATION_SEEDS, flush=True)

def experiments_for_evaluation(phase: str) -> List[str]:
    if not CFG.run_full_paper:
        return ["UniGDNetR"]
    choices = {
        "main": ["UniGDNetR"],
        "ablations": [
            name for name in EXPERIMENTS
            if name in {"Uniform-slice", "Shared-no-FiLM", "No-boundary"}
        ],
        "single-task": [name for name in EXPERIMENTS if name.startswith("SingleTask-")],
        "all": list(EXPERIMENTS),
    }
    if phase not in choices:
        raise ValueError(f"Unknown EVALUATION_PHASE={phase!r}; choose one of {sorted(choices)}")
    return choices[phase]


trained_checkpoints = []
checkpoint_rows = []
missing_or_invalid = []

for experiment_name in experiments_for_evaluation(EVALUATION_PHASE):
    kwargs = EXPERIMENTS[experiment_name]
    for seed in EVALUATION_SEEDS:
        checkpoint_path = OUTPUT_ROOT / "checkpoints" / f"{experiment_name}_seed{seed}.pt"
        status = "missing"
        reason = "file not found"
        if checkpoint_path.exists():
            try:
                checkpoint = torch_load_compat(checkpoint_path, map_location="cpu")
                reasons = checkpoint_metadata_reasons(
                    checkpoint,
                    experiment_name,
                    seed,
                    kwargs,
                    require_complete=True,
                )
                if reasons:
                    status = "invalid"
                    reason = "; ".join(reasons)
                else:
                    status = "ready"
                    reason = "compatible completed checkpoint"
                    trained_checkpoints.append(checkpoint_path)
            except Exception as exc:
                status = "invalid"
                reason = f"{type(exc).__name__}: {exc}"

        checkpoint_rows.append({
            "experiment": experiment_name,
            "seed": int(seed),
            "status": status,
            "reason": reason,
            "path": str(checkpoint_path),
        })
        if status != "ready":
            missing_or_invalid.append(f"{experiment_name}/seed{seed}: {reason}")

checkpoint_inventory_df = pd.DataFrame(checkpoint_rows)
display(checkpoint_inventory_df)

if missing_or_invalid and REQUIRE_ALL_CHECKPOINTS:
    raise RuntimeError(
        "Evaluation is blocked because required checkpoints are unavailable:\n- "
        + "\n- ".join(missing_or_invalid)
        + "\nFinish Stage 2 training, or set REQUIRE_ALL_CHECKPOINTS=False only for a labelled partial diagnostic run."
    )
if not trained_checkpoints:
    raise RuntimeError("No compatible completed checkpoint is available for evaluation.")

trained_checkpoints = sorted(trained_checkpoints, key=lambda path: path.name)
print(f"Evaluation will use {len(trained_checkpoints)} verified checkpoint(s).")

In [ ]:
# Job-level evaluation cache. Every key includes the corrected data fingerprint,
# pipeline revision, checkpoint identity, and evaluation arguments. Interrupted
# evaluation therefore resumes at the next unfinished job without mixing revisions.
EVALUATION_CACHE_REVISION = "2026-10-25-job-cache-v4-brats-rc-patient"
EVAL_CACHE_ROOT = OUTPUT_ROOT / "results" / "evaluation_cache"
EVAL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)


def _checkpoint_file_signature(path: Path) -> dict:
    stat = path.stat()
    return {"path": str(path.resolve()), "size": int(stat.st_size), "mtime_ns": int(stat.st_mtime_ns)}


def _evaluation_cache_key(kind: str, checkpoint_path: Path, task: str, **parameters) -> str:
    payload = {
        "cache_revision": EVALUATION_CACHE_REVISION,
        "pipeline_revision": PIPELINE_REVISION,
        "dataset_fingerprint": DATASET_FINGERPRINT,
        "checkpoint": _checkpoint_file_signature(checkpoint_path),
        "kind": kind,
        "task": task,
        "parameters": parameters,
    }
    serialised = json.dumps(payload, sort_keys=True, separators=(",", ":"), default=str)
    return hashlib.sha256(serialised.encode("utf-8")).hexdigest()


def _atomic_dataframe_save(frame: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + f".{os.getpid()}.tmp")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def cached_temperature(model, checkpoint_path: Path, checkpoint: dict, task: str) -> float:
    cache_key = _evaluation_cache_key(
        "temperature", checkpoint_path, task,
        seed=int(checkpoint["seed"]), max_batches=80 if CFG.run_full_paper else 5,
    )
    path = EVAL_CACHE_ROOT / f"temperature_{cache_key}.json"
    if path.exists():
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
            value = float(payload["temperature"])
            if np.isfinite(value) and 0.05 <= value <= 10.0:
                print("Using cached temperature:", checkpoint["experiment"], checkpoint["seed"], task)
                return value
        except Exception:
            pass

    task_offset = sum((index + 1) * ord(char) for index, char in enumerate(task))
    calibration_seed = int(checkpoint["seed"]) * 1009 + task_offset
    value = fit_temperature(
        model,
        loaders[task]["val"],
        task,
        max_batches=80 if CFG.run_full_paper else 5,
        sampling_seed=calibration_seed,
    )
    _atomic_json_save({"temperature": value, "cache_key": cache_key}, path)
    return value


def cached_evaluate_records(model, checkpoint_path: Path, checkpoint: dict, records: List[dict],
                            task: str, experiment: str, seed: int, temperature: float,
                            mc_samples: int, missing_modalities=None,
                            corruption=("clean", 0.0)) -> pd.DataFrame:
    cache_key = _evaluation_cache_key(
        "volume_metrics", checkpoint_path, task,
        experiment=experiment,
        seed=int(seed),
        temperature=float(temperature),
        mc_samples=int(mc_samples),
        missing_modalities=list(missing_modalities or []),
        corruption=list(corruption),
        case_ids=[record["case_id"] for record in records],
    )
    path = EVAL_CACHE_ROOT / f"metrics_{cache_key}.csv"
    required = {
        "experiment", "seed", "task", "case_id", "patient_id", "region",
        "dice", "hd95", "ece",
    }
    expected_cases = {record["case_id"] for record in records}
    expected_case_patient_pairs = {
        (
            str(record["case_id"]),
            str(record.get("patient_id", record.get("group_id", record["case_id"]))),
        )
        for record in records
    }
    if task == "brats":
        expected_regions = {"WT", "TC", "NETC", "SNFH", "ET", "RC"}
    else:
        expected_regions = set(TASK_SPECS[task]["class_names"][1:])
    if path.exists():
        try:
            cached = pd.read_csv(path)
            cache_valid = (
                required.issubset(cached.columns)
                and set(cached["case_id"].astype(str)) == expected_cases
                and set(zip(
                    cached["case_id"].astype(str), cached["patient_id"].astype(str)
                )) == expected_case_patient_pairs
                and set(cached["region"].astype(str)) == expected_regions
                and not cached.duplicated(["case_id", "region"]).any()
                and len(cached) == len(expected_cases) * len(expected_regions)
                and set(cached["experiment"].astype(str)) == {str(experiment)}
                and set(cached["seed"].astype(int)) == {int(seed)}
                and set(cached["task"].astype(str)) == {str(task)}
            )
            if cache_valid:
                print("Using cached evaluation job:", experiment, seed, task, missing_modalities, corruption)
                return cached
            print("Ignoring incomplete or inconsistent evaluation cache:", path.name)
        except Exception as exc:
            print("Ignoring unreadable evaluation cache:", path.name, type(exc).__name__)

    # Make Gaussian corruption and MC-dropout sampling reproducible per job,
    # then restore the caller's RNG so cache hits/misses cannot change later jobs.
    saved_rng = _rng_state()
    job_seed = int(cache_key[:8], 16)
    seed_everything(job_seed)
    try:
        result = evaluate_records(
            model, records, task, experiment, seed, temperature, mc_samples,
            missing_modalities=missing_modalities, corruption=corruption,
        )
    finally:
        _restore_rng_state(saved_rng)
    if result.empty:
        raise RuntimeError(f"Evaluation produced no rows: {experiment}/seed{seed}/{task}")
    _atomic_dataframe_save(result, path)
    return result

## 15. Clean volumetric test evaluation


In [ ]:
all_clean_results = []
temperature_rows = []
for ckpt_path in trained_checkpoints:
    net, ckpt = load_checkpoint(ckpt_path)
    experiment, seed = ckpt["experiment"], ckpt["seed"]
    for task in ckpt["task_specs"]:
        temperature = cached_temperature(net, ckpt_path, ckpt, task)
        temperature_rows.append({"experiment": experiment, "seed": seed, "task": task, "temperature": temperature})
        test_records = splits[task]["test"]
        if not CFG.run_full_paper:
            test_records = test_records[:1]
        result = cached_evaluate_records(
            net, ckpt_path, ckpt, test_records, task, experiment, seed, temperature,
            mc_samples=CFG.mc_dropout_samples if CFG.run_full_paper else 1,
        )
        all_clean_results.append(result)
    del net
    if torch.cuda.is_available(): torch.cuda.empty_cache()

temperatures_df = pd.DataFrame(temperature_rows)
clean_results_df = pd.concat(all_clean_results, ignore_index=True)
_atomic_dataframe_save(temperatures_df, OUTPUT_ROOT / "results" / "temperatures.csv")
_atomic_dataframe_save(clean_results_df, OUTPUT_ROOT / "results" / "clean_case_metrics.csv")
display(clean_results_df.groupby(["experiment","task","region"])[["dice","hd95","ece"]].agg(["mean","std"]))

## 16. BraTS missing-modality robustness

The benchmark reports complete input, each single missing sequence, and each single available sequence.
It runs only for models containing the BraTS task.


In [ ]:
def brats_missing_scenarios():
    names = ["T1n","T1c","T2w","T2-FLAIR"]
    scenarios = {"all": []}
    for i, name in enumerate(names):
        scenarios[f"missing_{name}"] = [i]
        scenarios[f"only_{name}"] = [j for j in range(4) if j != i]
    return scenarios

missing_results = []
if CFG.run_full_paper and "brats" in TASK_SPECS:
    for ckpt_path in trained_checkpoints:
        net, ckpt = load_checkpoint(ckpt_path)
        if "brats" not in ckpt["task_specs"]:
            del net
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            continue
        exp_name, run_seed = ckpt["experiment"], ckpt["seed"]
        temp = temperatures_df.query("experiment == @exp_name and seed == @run_seed and task == 'brats'")["temperature"].iloc[0]
        for scenario, missing in brats_missing_scenarios().items():
            df = cached_evaluate_records(
                net, ckpt_path, ckpt, splits["brats"]["test"], "brats",
                ckpt["experiment"], ckpt["seed"], temp, CFG.mc_dropout_samples,
                missing_modalities=missing,
            )
            df["scenario"] = scenario
            missing_results.append(df)
        del net
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    if not missing_results:
        raise RuntimeError("No BraTS missing-modality results were produced.")
    missing_df = pd.concat(missing_results, ignore_index=True)
    _atomic_dataframe_save(missing_df, OUTPUT_ROOT / "results" / "brats_missing_modality_metrics.csv")
    display(missing_df.groupby(["experiment","scenario","region"])["dice"].mean().unstack(-1))
else:
    print("Enable CFG.run_full_paper to run the missing-modality benchmark.")

## 17. Acquisition-shift robustness benchmark

These corruptions are not claimed to reproduce every scanner shift. They form a controlled stress test
and should be complemented by a genuinely external dataset if one becomes available.


In [ ]:
CORRUPTIONS = [
    ("clean", 0.0),
    ("gaussian_noise", 0.05), ("gaussian_noise", 0.10), ("gaussian_noise", 0.20),
    ("blur", 3), ("blur", 5),
    ("resolution", 0.75), ("resolution", 0.50),
    ("gamma", 0.75), ("gamma", 1.50),
]

corruption_results = []
if CFG.run_full_paper:
    main_ckpts = [p for p in trained_checkpoints if p.name.startswith("UniGDNetR_")]
    for ckpt_path in main_ckpts:
        net, ckpt = load_checkpoint(ckpt_path)
        run_seed = ckpt["seed"]
        for task in ckpt["task_specs"]:
            temp = temperatures_df.query("experiment == 'UniGDNetR' and seed == @run_seed and task == @task")["temperature"].iloc[0]
            for corruption in CORRUPTIONS:
                df = cached_evaluate_records(
                    net, ckpt_path, ckpt, splits[task]["test"], task, "UniGDNetR",
                    ckpt["seed"], temp, CFG.mc_dropout_samples, corruption=corruption,
                )
                corruption_results.append(df)
        del net
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    if not corruption_results:
        raise RuntimeError("No acquisition-shift results were produced.")
    corruption_df = pd.concat(corruption_results, ignore_index=True)
    _atomic_dataframe_save(corruption_df, OUTPUT_ROOT / "results" / "corruption_metrics.csv")
else:
    print("Enable CFG.run_full_paper to run the acquisition-shift benchmark.")

## 18. Statistical analysis

The unit of analysis is the patient/case, not the slice. Confidence intervals are bootstrapped over
cases. Pairwise tests use paired Wilcoxon signed-rank tests followed by Holm correction.


In [ ]:
def bootstrap_ci(values, statistic=np.nanmean, n_boot=10000, alpha=0.05, seed=13):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < 2:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    stats = np.array([statistic(rng.choice(x, len(x), replace=True)) for _ in range(n_boot)])
    return float(statistic(x)), float(np.quantile(stats, alpha/2)), float(np.quantile(stats, 1-alpha/2))

def holm_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    adjusted = np.empty_like(p)
    running = 0.0
    m = len(p)
    for rank, idx in enumerate(order):
        value = min(1.0, (m-rank)*p[idx])
        running = max(running, value)
        adjusted[idx] = running
    return adjusted

patient_level_results_df = (
    clean_results_df
    .groupby(["experiment", "task", "region", "patient_id"], as_index=False)
    .agg(
        dice=("dice", "mean"),
        hd95=("hd95", "mean"),
        ece=("ece", "mean"),
        seeds_observed=("seed", "nunique"),
        sessions_observed=("case_id", "nunique"),
    )
)
_atomic_dataframe_save(
    patient_level_results_df,
    OUTPUT_ROOT / "results" / "clean_patient_metrics_seed_and_session_averaged.csv",
)

summary_rows = []
for keys, group in patient_level_results_df.groupby(["experiment", "task", "region"]):
    mean, lo, hi = bootstrap_ci(group["dice"], n_boot=10000 if CFG.run_full_paper else 200)
    summary_rows.append({
        "experiment": keys[0], "task": keys[1], "region": keys[2],
        "dice_mean": mean, "dice_ci_low": lo, "dice_ci_high": hi,
        "n_patients": group["patient_id"].nunique(),
        "median_sessions_per_patient": float(group["sessions_observed"].median()),
        "median_seeds_per_patient": float(group["seeds_observed"].median()),
    })
summary_df = pd.DataFrame(summary_rows)
_atomic_dataframe_save(summary_df, OUTPUT_ROOT / "tables" / "table_primary_dice_bootstrap_ci.csv")
display(summary_df)

def paired_tests(results: pd.DataFrame, reference="UniGDNetR") -> pd.DataFrame:
    rows = []
    for (task, region), subset in results.groupby(["task","region"]):
        pivot = subset.pivot_table(index="patient_id", columns="experiment", values="dice", aggfunc="mean")
        if reference not in pivot:
            continue
        for comparator in [c for c in pivot.columns if c != reference]:
            pair = pivot[[reference, comparator]].dropna()
            if len(pair) < 5 or np.allclose(pair[reference], pair[comparator]):
                continue
            stat, p = wilcoxon(pair[reference], pair[comparator], alternative="two-sided")
            rows.append({"task":task,"region":region,"reference":reference,"comparator":comparator,
                         "n_pairs":len(pair),"mean_difference":float((pair[reference]-pair[comparator]).mean()),
                         "wilcoxon_statistic":stat,"p_raw":p})
    out = pd.DataFrame(rows)
    if len(out):
        out["p_holm"] = holm_adjust(out["p_raw"].values)
    return out

tests_df = paired_tests(patient_level_results_df)
_atomic_dataframe_save(tests_df, OUTPUT_ROOT / "tables" / "table_paired_wilcoxon_holm.csv")
display(tests_df)

## 19. Figures and architecture behaviour


In [ ]:
sns.set_theme(style="whitegrid", context="talk")
if len(clean_results_df):
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=patient_level_results_df, x="task", y="dice", hue="experiment")
    plt.ylim(0, 1.02)
    plt.title("Patient-level positive-region Dice")
    plt.tight_layout()
    plt.savefig(OUTPUT_ROOT / "figures" / "figure_primary_dice_boxplot.png", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
@torch.no_grad()
def profile_runtime(model, task: str, repeats=100):
    spec = TASK_SPECS[task]
    x = torch.randn(1, spec["modalities"]*CFG.context_slices, CFG.image_size, CFG.image_size, device=DEVICE)
    model.eval()
    for _ in range(10):
        _ = model(x, task)
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(repeats):
        _ = model(x, task)
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    ms = (time.perf_counter()-start)*1000/repeats
    memory = torch.cuda.max_memory_allocated()/1024**2 if DEVICE.type == "cuda" else np.nan
    return {"task":task,"parameters":count_parameters(model),"milliseconds_per_slice":ms,"peak_cuda_mb":memory}

if trained_checkpoints:
    main_profile_paths = [path for path in trained_checkpoints if path.name.startswith("UniGDNetR_")]
    profile_path = main_profile_paths[0] if main_profile_paths else trained_checkpoints[0]
    profile_net, _ = load_checkpoint(profile_path)
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    profile_df = pd.DataFrame([
        profile_runtime(profile_net, task, repeats=100 if CFG.run_full_paper else 10)
        for task in profile_net.task_specs
    ])
    _atomic_dataframe_save(profile_df, OUTPUT_ROOT / "tables" / "table_efficiency.csv")
    display(profile_df)
    del profile_net
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 20. Optional official nnU-Net v2 baseline export and commands

A Q1 paper should include nnU-Net v2 on identical patient splits. The function below exports one task
into nnU-Net format. It copies data, so ensure adequate disk space. Install the official `nnunetv2`
package in a separate environment if possible.


In [ ]:
import shutil

def export_nnunet_dataset(task: str, records: List[dict], dataset_id: int, dataset_name: str,
                          raw_root: Path, overwrite=False):
    if not records:
        raise ValueError(f"No records supplied for nnU-Net export: {task}")
    target = raw_root / f"Dataset{dataset_id:03d}_{dataset_name}"
    if target.exists():
        if not overwrite:
            print("Exists:", target)
            return target
        backup = target.with_name(target.name + "_backup_" + time.strftime("%Y%m%d_%H%M%S"))
        target.replace(backup)
        print("Moved previous export to:", backup)
    (target / "imagesTr").mkdir(parents=True, exist_ok=False)
    (target / "labelsTr").mkdir(parents=True, exist_ok=False)
    channel_names = {str(i): name for i, name in enumerate(records[0]["modalities"])}
    case_map = {}

    for i, record in enumerate(tqdm(records, desc=f"Export nnU-Net {task}")):
        new_id = f"{task}_{i:04d}"
        case_map[record["case_id"]] = new_id
        reference = nib.as_closest_canonical(nib.load(record["images"][0]))
        for modality, image_path in enumerate(record["images"]):
            image = nib.as_closest_canonical(nib.load(image_path))
            if not _same_grid(image, reference):
                image = resample_from_to(image, reference, order=1)
            nib.save(image, target / "imagesTr" / f"{new_id}_{modality:04d}.nii.gz")

        mask_image = nib.as_closest_canonical(nib.load(record["mask"]))
        if not _same_grid(mask_image, reference):
            mask_image = resample_from_to(mask_image, reference, order=0)
        mask = np.asarray(mask_image.dataobj, dtype=np.float32)
        if mask.ndim == 4 and mask.shape[-1] == 1:
            mask = mask[..., 0]
        mask = standardise_mask_labels(
            mask,
            task,
            record["mask"],
            brats_label_schema=record.get("brats_label_schema"),
        ).astype(np.uint8)
        label_image = nib.Nifti1Image(mask, reference.affine)
        nib.save(label_image, target / "labelsTr" / f"{new_id}.nii.gz")

    labels = {name: i for i, name in enumerate(TASK_SPECS[task]["class_names"])}
    dataset_json = {
        "channel_names": channel_names,
        "labels": labels,
        "numTraining": len(records),
        "file_ending": ".nii.gz",
    }
    (target / "dataset.json").write_text(json.dumps(dataset_json, indent=2), encoding="utf-8")
    (target / "case_map.json").write_text(json.dumps(case_map, indent=2), encoding="utf-8")
    return target

print("Suggested commands after export:")
print("pip install nnunetv2")
print("nnUNetv2_plan_and_preprocess -d 501 --verify_dataset_integrity")
print("nnUNetv2_train 501 2d 0")


## 21. Full-paper execution matrix

Before submission, the evidence package should contain:

| Analysis | Required output |
|---|---|
| Main model | 3 seeds × 3 datasets, volumetric Dice/HD95/sensitivity/precision/ECE |
| Strong baseline | nnU-Net v2 on identical patient splits |
| Single-task control | one GDNet per dataset |
| Sharing ablation | shared core without FiLM |
| Slice-fusion ablation | uniform instead of learned neighbouring-slice weights |
| Loss ablation | without boundary consistency |
| Robustness | noise, blur, resolution and gamma shifts |
| Missing MRI | all single-missing and single-available BraTS sequences |
| Reliability | temperature-scaled ECE, uncertainty-error association, risk-coverage curves |
| Efficiency | parameters, peak VRAM, training time and inference latency |
| Statistics | patient-level bootstrap CIs, paired Wilcoxon, Holm-adjusted p-values |

Do not write superiority claims until these analyses are complete. A negative multi-task result is
still publishable if the study rigorously identifies when cross-anatomy sharing helps or causes
negative transfer.


## 22. Journal strategy

- **Most realistic first target:** *Computers in Biology and Medicine* or *Computer Methods and
  Programs in Biomedicine*, provided all three datasets, nnU-Net v2, robustness, calibration and
  statistical analyses are complete.
- **Higher-risk/high-prestige target:** *Medical Image Analysis*. This requires a clear method advance,
  multiple external datasets, strong contemporary baselines and convincing improvements—not only a
  new module name.
- **Alternative clinical-AI target:** *Artificial Intelligence in Medicine* if the paper emphasises
  reliability, deployment constraints and clinical failure analysis.

Journals do not generally publish reliable method-specific acceptance probabilities. Verify current
quartile and scope at submission because rankings change by year and category.

Key literature anchors:

1. Ma et al., “Segment anything in medical images,” *Nature Communications* (MedSAM), 2024.
2. Isensee et al., “nnU-Net Revisited: A Call for Rigorous Validation in 3D Medical Image Segmentation,” 2024.
3. Zhao et al., “Large-vocabulary segmentation for medical images with text prompts,” *npj Digital Medicine*, 2025.
4. Recent work on probabilistic/generalizable segmentation and incomplete-modality segmentation should
   be reviewed immediately before manuscript submission.


## 23. Reproducibility record


In [ ]:
reproducibility = {
    "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
    "pipeline_revision": PIPELINE_REVISION,
    "dataset_fingerprint": DATASET_FINGERPRINT,
    "evaluation_cache_revision": EVALUATION_CACHE_REVISION,
    "statistical_unit": "patient; longitudinal sessions and repeated seeds averaged before inference",
    "task_dataset_fingerprints": TASK_DATASET_FINGERPRINTS,
    "platform": platform.platform(),
    "python": sys.version,
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "cpu_count": psutil.cpu_count(logical=True),
    "ram_gb": round(psutil.virtual_memory().total/1024**3, 2),
    "config": asdict(CFG),
    "task_specs": TASK_SPECS,
    "dataset_counts": {k: len(v) for k, v in all_records.items()},
    "checkpoint_files": [str(path) for path in trained_checkpoints],
}
_atomic_json_save(
    reproducibility,
    OUTPUT_ROOT / "results" / "reproducibility.json",
)
print(json.dumps(reproducibility, indent=2))

In [ ]:
# Atomic completion marker for automated or unattended runs.
evaluation_completion = {
    "automatic_evaluation": True,
    "phase": EVALUATION_PHASE,
    "seeds": [int(seed) for seed in EVALUATION_SEEDS],
    "checkpoint_count": len(trained_checkpoints),
    "checkpoint_files": [str(path) for path in trained_checkpoints],
    "completed_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "pipeline_revision": PIPELINE_REVISION,
    "dataset_fingerprint": DATASET_FINGERPRINT,
}
evaluation_completion_path = OUTPUT_ROOT / "results" / "automatic_evaluation_complete.json"
_atomic_json_save(evaluation_completion, evaluation_completion_path)

print("FULL AUTOMATIC EVALUATION COMPLETE", flush=True)
print("Completion record:", evaluation_completion_path, flush=True)